
# Sentinel-1 SAR Ship Detection V4.1 — FINAL KAGGLE NOTEBOOK

This notebook implements the complete controlled ship-detection experiment:

**Sentinel-1 VV/VH → dual YOLO candidate detection → E0/E1 fusion → land/coast evidence → SAR target segmentation and apparent geometry → AIS association → development-only persistence → conservative five-class evidence fusion → maps, tables, GeoJSON, QA plots and export ZIPs.**

Final object classes:

1. `AIS_MATCHED_VESSEL`
2. `NON_AIS_LIKELY_VESSEL`
3. `STATIC_MARITIME_OBJECT`
4. `PROBABLE_FALSE_ALARM`
5. `UNCERTAIN`

Important scientific conventions:

- YOLO detections are **candidates**, not final ships.
- Land is **not hard-masked before YOLO**. Land/coast evidence is evaluated after detection so genuine nearshore vessels are retained.
- `sar_apparent_length_m`, `sar_apparent_width_m`, and `sar_signature_area_m2` describe the **apparent SAR signature**, not guaranteed physical hull dimensions.
- Development scenes derive/freeze thresholds and persistence. Holdout scenes must remain untouched until those rules are frozen.



## How to run this notebook in Kaggle

### First scientific run: development only

1. Create a Kaggle notebook and enable **GPU**.
2. Enable Internet if Sentinel-1/GSHHG/packages must be downloaded.
3. Attach the two frozen model files:
   - `E0_YOLO11s_SSDD_Baseline.pt`
   - `E1_YOLO11s_SSDD_SensorMix.pt`
4. Keep:
   - `PROCESS_HOLDOUT = False`
   - `ALLOW_PREVIOUS_CHECKPOINT_REUSE = False` for a clean rerun.
5. Run **all cells from top to bottom**.
6. Provide Earthdata credentials when requested.
7. Add the Global Fishing Watch token as Kaggle secret `fish`, or enter it when prompted.

The development run creates the V4.1 frozen thresholds, final classification maps, AIS detector audit, target-property QA plots, GeoJSON, final result tables and restart checkpoint ZIP.

### Holdout run

Only after the development results are reviewed and `FROZEN_THRESHOLDS_V4_1.json` exists:

1. Keep the same Kaggle session, or attach the V4.1 persistence/checkpoint output.
2. If using an attached previous V4.1 checkpoint, set `ALLOW_PREVIOUS_CHECKPOINT_REUSE = True`.
3. Set `PROCESS_HOLDOUT = True`.
4. Do **not** modify the frozen classification thresholds after viewing holdout results.
5. Rerun from the top. Existing development checkpoints are reused and the missing holdout scenes are added.



## V4.1 execution rule

For the clean rerun requested here, the working folder is versioned as `SAR_Ship_Project_V4_1`, so old V4 outputs cannot silently replace V4.1 outputs.

The first run must use `PROCESS_HOLDOUT = False`.

Review development outputs before unlocking holdout, especially:

- known island/coast false detections,
- known coastal vessels,
- `AIS_CENTERED_DETECTOR_AUDIT.csv`,
- final five-class overlays,
- AIS-vs-SAR apparent-length QA,
- target segmentation quality,
- persistence/static clusters,
- threshold stability.

Only after the development rules are frozen should the three holdout scenes be processed.


In [ ]:
# STEP 0 - Install only packages that may be missing in Kaggle.
# Internet: required only if Kaggle does not already have these packages cached.
%pip install -q ultralytics asf_search geopandas pyarrow shapely pyproj rasterio scikit-learn scipy opencv-python-headless

In [ ]:
# STEP 1 - Imports and environment audit
from __future__ import annotations

import os
import sys
import json
import math
import time
import shutil
import hashlib
import zipfile
import warnings
import platform
import traceback
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import rasterio
import requests
import geopandas as gpd
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union, transform as shapely_transform
from pyproj import Transformer
from scipy.interpolate import griddata
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import DBSCAN
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

import asf_search as asf
from ultralytics import YOLO

warnings.filterwarnings('ignore', category=UserWarning)

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# STEP 2 - Master configuration
# Change values only here. Do not hardcode thresholds later in the notebook.

WORK_ROOT = Path('/kaggle/working/SAR_Ship_Project_V4_1')
KAGGLE_INPUT = Path('/kaggle/input')

RUN = {
    'scene_catalog': True,
    'sar_preprocessing': True,
    'yolo_detection': True,
    'candidate_fusion': True,
    'spatial_context': True,
    'sar_features': True,
    'ais': True,
    'temporal': True,
    'classification': True,
    'validation': True,
    'hard_negatives': True,
    'export': True,
}

FORCE = {k: False for k in RUN}

# Protect the holdout. Leave False until development thresholds are frozen.
PROCESS_HOLDOUT = False

# Clean rerun default. Set True only when intentionally attaching a V4.1 checkpoint.
ALLOW_PREVIOUS_CHECKPOINT_REUSE = False

# Debug runs are never final results.
DEBUG_MODE = False
DEBUG_SCENE_IDS = []   # e.g. ['S03_20240626']

# Keep paper-style output compact by default.
MAKE_DIAGNOSTIC_FIGURES = True
EXPORT_HARD_NEGATIVE_CHIPS = False
MAKE_FINAL_CLASSIFICATION_MAPS = True
MAKE_TARGET_PROPERTY_PLOTS = True
EXPORT_FINAL_GEOJSON = True
DELETE_TEMP_SAFE_AFTER_CACHE = True

CFG = {
    'project_version': 'V4_1_coastal_audit',
    'lon_min': 103.55,
    'lon_max': 104.15,
    'lat_min': 1.05,
    'lat_max': 1.30,
    'utm_epsg': 32648,
    'search_windows': {
        2024: ('2024-06-01T00:00:00Z', '2024-08-31T23:59:59Z'),
        2025: ('2025-06-01T00:00:00Z', '2025-08-31T23:59:59Z'),
        2026: ('2026-06-01T00:00:00Z', '2026-08-31T23:59:59Z'),
    },
    'expected_dates': [
        '2024-06-02','2024-06-14','2024-06-26','2024-07-08','2024-07-20','2024-08-01','2024-08-13','2024-08-25',
        '2025-06-09','2025-06-21','2025-07-03','2025-07-15','2025-07-27','2025-08-08','2025-08-20',
        '2026-06-04','2026-06-16','2026-06-28',
    ],
    'holdout_dates': ['2024-08-13','2025-08-20','2026-06-28'],
    'platform_prefix': 'S1A_',
    'flight_direction': 'ASCENDING',
    'relative_path': 171,
    'beam_mode': 'IW',
    'polarization': 'VV+VH',
    'tile_size': 640,
    'tile_stride': 512,
    'conf_threshold': 0.25,  # candidate-generation threshold, not final vessel threshold
    'tile_nms_iou': 0.50,
    'global_nms_iou': 0.50,
    'crop_buffer_px': 100,
    'db_min': -30.0,
    'db_max': 5.0,
    'pixel_spacing_m_approx': 10.0,
    # E0/E1 fusion
    'fusion_max_center_px': 25.0,
    'fusion_min_iou': 0.05,
    # coast zones
    'coast_zone_1_m': 300.0,
    'coast_zone_2_m': 1000.0,
    # AIS
    'gfw_report_url': 'https://gateway.api.globalfishingwatch.org/v3/4wings/report',
    'gfw_dataset': 'public-global-presence:latest',
    'ais_query_window_hours': 2.0,
    'ais_nearest_point_max_minutes': 45.0,
    'ais_hard_match_max_m': 800.0,
    # Provisional AIS-distance gates. Validate with reviewed pairs before publication.
    'ais_interp_high_m': 300.0,
    'ais_interp_moderate_m': 600.0,
    'ais_nearest_high_m': 250.0,
    'ais_nearest_moderate_m': 450.0,
    # persistence
    'persistence_eps_m': 90.0,
    'persistence_min_samples': 3,
    'static_min_unique_dates': 5,
    'static_min_unique_years': 2,
    'static_median_dispersion_max_m': 40.0,
    'static_max_dispersion_max_m': 90.0,
    # apparent SAR-signature plausibility, deliberately broad
    'min_apparent_length_m': 15.0,
    'max_apparent_length_m': 450.0,
    'min_apparent_width_m': 5.0,
    'max_apparent_width_m': 140.0,
    'min_sar_aspect_ratio': 1.15,
    'max_sar_aspect_ratio': 18.0,
    # engineering land-conflict thresholds
    'moderate_bbox_land_overlap': 0.15,
    'moderate_target_land_overlap': 0.20,
    'strong_bbox_land_overlap': 0.50,
    'strong_target_land_overlap': 0.50,
    'coastal_low_land_bbox_max': 0.15,
    'coastal_low_land_target_max': 0.20,
    'static_candidate_centroid_max_m': 45.0,
}

AOI_WKT = (
    f"POLYGON(({CFG['lon_min']} {CFG['lat_min']}, {CFG['lon_max']} {CFG['lat_min']}, "
    f"{CFG['lon_max']} {CFG['lat_max']}, {CFG['lon_min']} {CFG['lat_max']}, "
    f"{CFG['lon_min']} {CFG['lat_min']}))"
)
AOI_GEOJSON = {
    'type': 'Polygon',
    'coordinates': [[
        [CFG['lon_min'], CFG['lat_min']], [CFG['lon_max'], CFG['lat_min']],
        [CFG['lon_max'], CFG['lat_max']], [CFG['lon_min'], CFG['lat_max']],
        [CFG['lon_min'], CFG['lat_min']],
    ]],
}

DIRS = {
    'scene_catalog': WORK_ROOT/'00_scene_catalog',
    'scene_cache': WORK_ROOT/'01_scene_cache',
    'raw_detections': WORK_ROOT/'02_raw_detections',
    'fusion': WORK_ROOT/'03_candidate_fusion',
    'spatial': WORK_ROOT/'04_spatial_context',
    'sar_features': WORK_ROOT/'05_sar_features',
    'ais': WORK_ROOT/'06_ais',
    'temporal': WORK_ROOT/'07_temporal',
    'classification': WORK_ROOT/'08_classification',
    'validation': WORK_ROOT/'09_validation',
    'hard_negatives': WORK_ROOT/'10_hard_negatives',
    'figures': WORK_ROOT/'11_figures',
    'tables': WORK_ROOT/'12_tables',
    'logs': WORK_ROOT/'13_logs',
    'manifests': WORK_ROOT/'14_manifests',
    'final': WORK_ROOT/'FINAL_EXPORT',
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
(DIRS['raw_detections']/'per_scene').mkdir(parents=True, exist_ok=True)
(DIRS['ais']/'raw_reports').mkdir(parents=True, exist_ok=True)
(DIRS['ais']/'per_scene').mkdir(parents=True, exist_ok=True)
(DIRS['figures']/'png').mkdir(parents=True, exist_ok=True)
(DIRS['figures']/'pdf').mkdir(parents=True, exist_ok=True)

CLASS_ORDER = [
    'AIS_MATCHED_VESSEL',
    'NON_AIS_LIKELY_VESSEL',
    'STATIC_MARITIME_OBJECT',
    'PROBABLE_FALSE_ALARM',
    'UNCERTAIN',
]

print('WORK_ROOT:', WORK_ROOT)
print('PROCESS_HOLDOUT:', PROCESS_HOLDOUT)
if DEBUG_MODE:
    print('WARNING: DEBUG MODE ACTIVE. Debug outputs are not final experiment results.')

In [ ]:
# STEP 3 - Checkpoint discovery and restart helpers

def _checkpoint_project_version(root):
    cfg_path = Path(root) / '14_manifests' / 'CONFIG.json'
    if not cfg_path.exists():
        return None
    try:
        with open(cfg_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data.get('CFG', {}).get('project_version')
    except Exception:
        return None


def find_previous_checkpoint_root():
    """
    Reuse is opt-in and only a checkpoint with the same project_version is accepted.
    This prevents an older V4 dataset from silently feeding V4.1 downstream stages.
    """
    if not ALLOW_PREVIOUS_CHECKPOINT_REUSE:
        return None

    if not KAGGLE_INPUT.exists():
        return None

    candidates = []
    for p in KAGGLE_INPUT.rglob('LOCKED_SCENE_CATALOG.csv'):
        if p.parent.name != '00_scene_catalog':
            continue
        root = p.parent.parent
        if _checkpoint_project_version(root) == CFG['project_version']:
            candidates.append(root)

    if not candidates:
        return None

    candidates = sorted(
        candidates,
        key=lambda p: (
            'v4_1' not in str(p).lower(),
            'checkpoint' not in str(p).lower(),
            len(str(p))
        )
    )
    return candidates[0]


PREVIOUS_ROOT = find_previous_checkpoint_root()
print('Previous checkpoint root:', PREVIOUS_ROOT)


def read_path(relative):
    relative = Path(relative)
    local = WORK_ROOT / relative
    if local.exists():
        return local
    if PREVIOUS_ROOT is not None:
        old = PREVIOUS_ROOT / relative
        if old.exists():
            return old
    return local


def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=str)


def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_table(df, stem):
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(stem.with_suffix('.csv'), index=False)
    try:
        df.to_parquet(stem.with_suffix('.parquet'), index=False)
    except Exception as e:
        print('Parquet save warning:', e)


def load_table(relative_stem):
    stem = read_path(relative_stem)
    pq = stem.with_suffix('.parquet')
    csv = stem.with_suffix('.csv')
    if pq.exists():
        return pd.read_parquet(pq)
    if csv.exists():
        return pd.read_csv(csv)
    return None


def file_sha256(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def stage_complete(stage, input_rows=None, output_rows=None, scenes=None, checkpoint=None, errors=0):
    print('\n' + '='*88)
    print('STAGE COMPLETE:', stage)
    if input_rows is not None: print('Input rows:', input_rows)
    if output_rows is not None: print('Output rows:', output_rows)
    if scenes is not None: print('Scenes:', scenes)
    if checkpoint is not None: print('Checkpoint:', checkpoint)
    print('Errors:', errors)
    print('='*88)

save_json(DIRS['manifests']/'CONFIG.json', {'CFG': CFG, 'RUN': RUN, 'FORCE': FORCE, 'PROCESS_HOLDOUT': PROCESS_HOLDOUT, 'ALLOW_PREVIOUS_CHECKPOINT_REUSE': ALLOW_PREVIOUS_CHECKPOINT_REUSE})

In [ ]:
# STEP 3B - Fail-fast experiment sanity checks

assert CFG['tile_stride'] <= CFG['tile_size'], "Tile stride must not exceed tile size."
assert len(CFG['expected_dates']) == len(set(CFG['expected_dates'])), "Duplicate expected dates found."
assert set(CFG['holdout_dates']).issubset(set(CFG['expected_dates'])), "Holdout dates must be a subset of expected dates."
assert 0 < CFG['conf_threshold'] < 1
assert CFG['coast_zone_1_m'] < CFG['coast_zone_2_m']
assert 0 <= CFG['moderate_bbox_land_overlap'] < CFG['strong_bbox_land_overlap'] <= 1
assert 0 <= CFG['moderate_target_land_overlap'] < CFG['strong_target_land_overlap'] <= 1

if PROCESS_HOLDOUT:
    frozen = read_path('08_classification/FROZEN_THRESHOLDS_V4_1.json')
    if not frozen.exists():
        raise RuntimeError(
            "PROCESS_HOLDOUT=True but no frozen V4.1 development thresholds were found. "
            "Run the development experiment first with PROCESS_HOLDOUT=False."
        )

print("Configuration sanity checks: PASS")
print("Checkpoint reuse enabled:", ALLOW_PREVIOUS_CHECKPOINT_REUSE)
print("Previous compatible checkpoint:", PREVIOUS_ROOT)


# Part A - Frozen E0/E1 detector weights

The operational notebook does not retrain SSDD. It searches `/kaggle/input` for the two frozen weight files. Attach your Kaggle model dataset before running this section.

In [ ]:
# STEP 4 - Locate E0/E1 weights in Kaggle inputs
EXPECTED_MODELS = {
    'E0': 'E0_YOLO11s_SSDD_Baseline.pt',
    'E1': 'E1_YOLO11s_SSDD_SensorMix.pt',
}

def find_weight(filename):
    exact = list(KAGGLE_INPUT.rglob(filename)) if KAGGLE_INPUT.exists() else []
    if exact:
        return exact[0]
    # Fallback by identifying E0/E1 tokens in .pt filenames.
    pts = list(KAGGLE_INPUT.rglob('*.pt')) if KAGGLE_INPUT.exists() else []
    token = 'e0' if filename.startswith('E0_') else 'e1'
    fuzzy = [p for p in pts if token in p.name.lower()]
    return fuzzy[0] if len(fuzzy) == 1 else None

MODEL_PATHS = {k: find_weight(v) for k, v in EXPECTED_MODELS.items()}
for k, p in MODEL_PATHS.items():
    print(k, '->', p)

if any(p is None for p in MODEL_PATHS.values()):
    available = list(KAGGLE_INPUT.rglob('*.pt')) if KAGGLE_INPUT.exists() else []
    print('\nAvailable .pt files:')
    for p in available[:100]:
        print(' ', p)
    raise FileNotFoundError('Could not uniquely locate both E0 and E1 model weights. Attach the sar-ship-models Kaggle dataset.')

In [ ]:
# STEP 5 - Load models only once
model_e0 = YOLO(str(MODEL_PATHS['E0']))
model_e1 = YOLO(str(MODEL_PATHS['E1']))
print('Loaded frozen E0 and E1 models.')

# Part B - Locked controlled Sentinel-1 scene catalogue

Controlled geometry:

- Sentinel-1A only
- IW GRD high-resolution
- VV+VH
- ascending
- relative path 171
- Singapore Strait AOI

The expected catalogue contains 18 scenes. Three dates are locked holdouts and must not influence thresholds or persistence-model construction.

In [ ]:
# STEP 6 - Search or load the locked scene catalogue
CATALOG_STEM = Path('00_scene_catalog/LOCKED_SCENE_CATALOG')
existing_catalog = load_table(CATALOG_STEM)

if existing_catalog is not None and not FORCE['scene_catalog']:
    scene_catalog = existing_catalog.copy()
    print('Loaded locked catalogue from checkpoint:', read_path(CATALOG_STEM))
else:
    if not RUN['scene_catalog']:
        raise RuntimeError('Scene catalogue is missing and RUN[scene_catalog] is False.')

    all_results = []
    for year, (start, end) in CFG['search_windows'].items():
        r = asf.geo_search(
            platform=asf.PLATFORM.SENTINEL1,
            intersectsWith=AOI_WKT,
            start=start,
            end=end,
            beamMode=CFG['beam_mode'],
            processingLevel=asf.PRODUCT_TYPE.GRD_HD,
            polarization=CFG['polarization'],
            maxResults=200,
        )
        all_results.extend(list(r))
        print(year, 'search results:', len(r))

    rows = []
    for product in all_results:
        p = product.properties
        scene_name = p.get('sceneName')
        if not scene_name:
            continue
        rows.append({
            'scene_name': scene_name,
            'platform': p.get('platform'),
            'start_time': p.get('startTime'),
            'flight_direction': str(p.get('flightDirection') or '').upper(),
            'path_number': p.get('pathNumber'),
            'polarization': p.get('polarization'),
            'processing_level': p.get('processingLevel'),
            'url': p.get('url'),
        })
    cand = pd.DataFrame(rows)
    if cand.empty:
        raise RuntimeError('ASF search returned no candidate scenes.')
    cand['start_time'] = pd.to_datetime(cand['start_time'], utc=True, errors='coerce')
    cand['date'] = cand['start_time'].dt.strftime('%Y-%m-%d')
    cand['path_number'] = pd.to_numeric(cand['path_number'], errors='coerce')
    pol = cand['polarization'].astype(str).str.upper()

    strict = cand[
        cand['scene_name'].str.startswith(CFG['platform_prefix'], na=False)
        & (cand['flight_direction'] == CFG['flight_direction'])
        & (cand['path_number'] == CFG['relative_path'])
        & pol.str.contains('VV', na=False)
        & pol.str.contains('VH', na=False)
    ].copy()
    strict = strict.sort_values('start_time').drop_duplicates('scene_name').reset_index(drop=True)

    expected = set(CFG['expected_dates'])
    strict = strict[strict['date'].isin(expected)].copy()
    dup_dates = strict['date'][strict['date'].duplicated(keep=False)].tolist()
    if dup_dates:
        raise RuntimeError(f'More than one strict scene found on expected dates: {sorted(set(dup_dates))}. Resolve manually; do not silently pick one.')

    found_dates = set(strict['date'])
    missing = sorted(expected - found_dates)
    extra = sorted(found_dates - expected)
    if missing or extra or len(strict) != 18:
        print(strict[['scene_name','start_time','flight_direction','path_number','polarization']].to_string(index=False))
        raise RuntimeError(f'Controlled catalogue mismatch. Found={len(strict)}, missing={missing}, extra={extra}')

    strict = strict.sort_values('start_time').reset_index(drop=True)
    strict['scene_id'] = [f"S{i:02d}_{d.replace('-', '')}" for i, d in enumerate(strict['date'], start=1)]
    scene_catalog = strict
    save_table(scene_catalog, DIRS['scene_catalog']/'LOCKED_SCENE_CATALOG')
    save_json(DIRS['scene_catalog']/'scene_catalog_metadata.json', {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'criteria': {
            'platform': 'Sentinel-1A', 'beam_mode': 'IW', 'product': 'GRD_HD',
            'polarization': 'VV+VH', 'flight_direction': 'ASCENDING', 'relative_path': 171,
        },
        'expected_dates': CFG['expected_dates'],
    })

scene_catalog['start_time'] = pd.to_datetime(scene_catalog['start_time'], utc=True, errors='coerce')
scene_catalog['date'] = scene_catalog['start_time'].dt.strftime('%Y-%m-%d')
scene_catalog['experiment_set'] = np.where(scene_catalog['date'].isin(CFG['holdout_dates']), 'HOLDOUT', 'DEVELOPMENT')

assert len(scene_catalog) == 18
assert (scene_catalog['experiment_set'] == 'DEVELOPMENT').sum() == 15
assert (scene_catalog['experiment_set'] == 'HOLDOUT').sum() == 3
assert set(scene_catalog.loc[scene_catalog['experiment_set']=='HOLDOUT','date']) == set(CFG['holdout_dates'])

save_table(scene_catalog, DIRS['scene_catalog']/'LOCKED_SCENE_CATALOG')
print(scene_catalog[['scene_id','date','scene_name','experiment_set']].to_string(index=False))
stage_complete('LOCKED SCENE CATALOG', output_rows=len(scene_catalog), scenes=18, checkpoint=DIRS['scene_catalog'])

In [ ]:
# STEP 7 - Decide which scenes this run is allowed to touch
FROZEN_THRESHOLD_PATH = read_path('08_classification/FROZEN_THRESHOLDS_V4_1.json')

if PROCESS_HOLDOUT:
    if not FROZEN_THRESHOLD_PATH.exists():
        raise RuntimeError('PROCESS_HOLDOUT=True but frozen development thresholds do not exist. Run development first and freeze thresholds.')
    allowed_catalog = scene_catalog.copy()
else:
    allowed_catalog = scene_catalog[scene_catalog['experiment_set']=='DEVELOPMENT'].copy()

if DEBUG_MODE:
    if not DEBUG_SCENE_IDS:
        DEBUG_SCENE_IDS = allowed_catalog['scene_id'].head(1).tolist()
    allowed_catalog = allowed_catalog[allowed_catalog['scene_id'].isin(DEBUG_SCENE_IDS)].copy()
    print('DEBUG scenes:', allowed_catalog['scene_id'].tolist())

print('Scenes allowed in this run:', len(allowed_catalog))
print(allowed_catalog[['scene_id','date','experiment_set']].to_string(index=False))

# Part C - Sentinel-1 calibration and per-scene cache

This is expensive and restart-safe. Each scene is calibrated once and saved as compressed VV/VH arrays plus geolocation metadata. If the cache already exists, the scene is not downloaded again.

In [ ]:
# STEP 8 - Sentinel-1 SAFE helpers (adapted from the working V3 notebook)

def local_tag(element):
    return element.tag.split('}')[-1]


def child_text(element, name):
    for child in element:
        if local_tag(child) == name:
            return child.text
    return None


def read_geolocation_grid(annotation_xml):
    points = []
    for elem in ET.parse(annotation_xml).getroot().iter():
        if local_tag(elem) != 'geolocationGridPoint':
            continue
        pixel, line = child_text(elem, 'pixel'), child_text(elem, 'line')
        lat, lon = child_text(elem, 'latitude'), child_text(elem, 'longitude')
        if None in (pixel, line, lat, lon):
            continue
        points.append([float(pixel), float(line), float(lon), float(lat)])
    points = np.asarray(points, dtype=np.float64)
    if len(points) == 0:
        raise RuntimeError(f'No geolocation grid points found in {annotation_xml}')
    return points


def find_band_files(safe_dir, polarization):
    tag = f'-{polarization.lower()}-'
    tiff = next((f for f in (safe_dir/'measurement').glob('*.tif*') if tag in f.name.lower()), None)
    ann = next((f for f in (safe_dir/'annotation').glob('*.xml') if tag in f.name.lower()), None)
    cal = next((f for f in (safe_dir/'annotation'/'calibration').glob('calibration-*.xml') if tag in f.name.lower()), None)
    if not (tiff and ann and cal):
        raise RuntimeError(f'Missing {polarization.upper()} product files under {safe_dir}')
    return tiff, ann, cal


def calibrate_and_crop(tiff_path, annotation_xml, calibration_xml, lon_min, lon_max, lat_min, lat_max, buffer_px=100):
    geo_points = read_geolocation_grid(annotation_xml)
    geo_xy = geo_points[:, 2:4]  # lon, lat
    geo_pixel, geo_line = geo_points[:, 0], geo_points[:, 1]
    aoi_corners = np.array([[lon_min,lat_min],[lon_max,lat_min],[lon_max,lat_max],[lon_min,lat_max]], dtype=np.float64)

    corner_px = griddata(geo_xy, geo_pixel, aoi_corners, method='linear')
    corner_ln = griddata(geo_xy, geo_line, aoi_corners, method='linear')
    for arr, vals in [(corner_px, geo_pixel), (corner_ln, geo_line)]:
        if np.any(np.isnan(arr)):
            fallback = griddata(geo_xy, vals, aoi_corners, method='nearest')
            arr[np.isnan(arr)] = fallback[np.isnan(arr)]

    with rasterio.open(tiff_path) as src:
        img_w, img_h = src.width, src.height
    x_min = max(0, int(np.floor(corner_px.min())) - buffer_px)
    x_max = min(img_w, int(np.ceil(corner_px.max())) + buffer_px)
    y_min = max(0, int(np.floor(corner_ln.min())) - buffer_px)
    y_max = min(img_h, int(np.ceil(corner_ln.max())) + buffer_px)

    window = rasterio.windows.Window(col_off=x_min, row_off=y_min, width=x_max-x_min, height=y_max-y_min)
    with rasterio.open(tiff_path) as src:
        raw_crop = src.read(1, window=window)

    cal_lines, cal_pixels, cal_sigma = [], [], []
    for elem in ET.parse(calibration_xml).getroot().iter():
        if local_tag(elem) != 'calibrationVector':
            continue
        line_txt = child_text(elem, 'line')
        pixel_txt = child_text(elem, 'pixel')
        sigma_txt = child_text(elem, 'sigmaNought')
        if None in (line_txt, pixel_txt, sigma_txt):
            continue
        cal_lines.append(int(line_txt))
        cal_pixels.append(np.fromstring(pixel_txt, sep=' ', dtype=np.float64))
        cal_sigma.append(np.fromstring(sigma_txt, sep=' ', dtype=np.float64))
    if not cal_lines:
        raise RuntimeError(f'No calibration vectors found in {calibration_xml}')

    order = np.argsort(cal_lines)
    cal_lines = np.asarray(cal_lines)[order]
    cal_pixels = [cal_pixels[i] for i in order]
    cal_sigma = [cal_sigma[i] for i in order]
    crop_columns = np.arange(x_min, x_max, dtype=np.float64)
    sigma_horizontal = np.asarray([np.interp(crop_columns, px, sig) for px, sig in zip(cal_pixels, cal_sigma)], dtype=np.float32)

    raw_float = raw_crop.astype(np.float32)
    sigma0 = np.empty(raw_float.shape, dtype=np.float32)
    for local_row, global_row in enumerate(np.arange(y_min, y_max)):
        upper = np.searchsorted(cal_lines, global_row)
        if upper == 0:
            lut = sigma_horizontal[0]
        elif upper >= len(cal_lines):
            lut = sigma_horizontal[-1]
        else:
            lower = upper - 1
            l0, l1 = cal_lines[lower], cal_lines[upper]
            alpha = 0.0 if l1 == l0 else (global_row-l0)/(l1-l0)
            lut = (1-alpha)*sigma_horizontal[lower] + alpha*sigma_horizontal[upper]
        sigma0[local_row,:] = raw_float[local_row,:]**2 / (lut*lut + 1e-12)

    sigma0[raw_crop == 0] = np.nan
    sigma0_db = 10.0*np.log10(np.maximum(sigma0, 1e-12))
    sigma0_db[~np.isfinite(sigma0)] = np.nan
    display_db = np.clip(sigma0_db, CFG['db_min'], CFG['db_max'])
    model_image = (display_db-CFG['db_min'])/(CFG['db_max']-CFG['db_min'])*255.0
    model_image = np.nan_to_num(model_image, nan=0).astype(np.uint8)

    return {
        'sigma0_db': sigma0_db,
        'model_image': model_image,
        'x_min': x_min, 'x_max': x_max, 'y_min': y_min, 'y_max': y_max,
        'geo_points': geo_points,
    }


def build_pixel_to_lonlat(geo_points):
    """
    Build interpolation objects ONCE per scene.

    V4 used scipy.griddata() for every queried pixel, which repeatedly rebuilt
    interpolation work and made target-land sampling unnecessarily slow.
    This V4.1 implementation constructs linear + nearest interpolators once and
    supports both scalar and vector queries.
    """
    from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

    tie_xy = np.asarray(geo_points[:,0:2], dtype=np.float64)
    tie_lon = np.asarray(geo_points[:,2], dtype=np.float64)
    tie_lat = np.asarray(geo_points[:,3], dtype=np.float64)

    lon_linear = LinearNDInterpolator(tie_xy, tie_lon, fill_value=np.nan)
    lat_linear = LinearNDInterpolator(tie_xy, tie_lat, fill_value=np.nan)
    lon_nearest = NearestNDInterpolator(tie_xy, tie_lon)
    lat_nearest = NearestNDInterpolator(tie_xy, tie_lat)

    def pixel_to_lonlat(pixel, line):
        px = np.asarray(pixel, dtype=np.float64)
        ln = np.asarray(line, dtype=np.float64)
        scalar = (px.ndim == 0 and ln.ndim == 0)

        px = np.atleast_1d(px)
        ln = np.atleast_1d(ln)
        q = np.column_stack([px, ln])

        lon = np.asarray(lon_linear(q), dtype=float).reshape(-1)
        lat = np.asarray(lat_linear(q), dtype=float).reshape(-1)

        bad_lon = ~np.isfinite(lon)
        bad_lat = ~np.isfinite(lat)
        if bad_lon.any():
            lon[bad_lon] = np.asarray(lon_nearest(q[bad_lon]), dtype=float).reshape(-1)
        if bad_lat.any():
            lat[bad_lat] = np.asarray(lat_nearest(q[bad_lat]), dtype=float).reshape(-1)

        if scalar:
            return float(lon[0]), float(lat[0])
        return lon, lat

    return pixel_to_lonlat


In [ ]:
# STEP 9 - Cache/read helpers and Earthdata authentication
_ASF_SESSION = None

def get_asf_session():
    global _ASF_SESSION
    if _ASF_SESSION is not None:
        return _ASF_SESSION
    username = input('Earthdata username: ').strip()
    password = getpass('Earthdata password: ')
    _ASF_SESSION = asf.ASFSession().auth_with_creds(username, password)
    return _ASF_SESSION


def scene_cache_paths(scene_id):
    return DIRS['scene_cache']/f'{scene_id}_VVVH.npz', DIRS['scene_cache']/f'{scene_id}_metadata.json'


def locate_scene_cache(scene_id):
    npz_rel = Path('01_scene_cache')/f'{scene_id}_VVVH.npz'
    meta_rel = Path('01_scene_cache')/f'{scene_id}_metadata.json'
    npz = read_path(npz_rel)
    meta = read_path(meta_rel)
    return (npz, meta) if npz.exists() and meta.exists() else (None, None)


def load_scene_cache(scene_id):
    npz, meta = locate_scene_cache(scene_id)
    if npz is None:
        raise FileNotFoundError(f'No cache for {scene_id}')
    z = np.load(npz, allow_pickle=False)
    md = load_json(meta)
    return {
        'vv_db': z['vv_db'], 'vh_db': z['vh_db'], 'model_image': z['model_image'],
        'geo_points': z['geo_points'],
        **md,
    }


def download_extract(product, temp_root):
    scene_name = product.properties['sceneName']
    scene_dir = temp_root/scene_name
    safe_dirs = list(scene_dir.rglob('*.SAFE'))
    if safe_dirs:
        return safe_dirs[0]
    scene_dir.mkdir(parents=True, exist_ok=True)
    session = get_asf_session()
    product.download(path=str(scene_dir), session=session)
    zips = list(scene_dir.glob('*.zip'))
    if not zips:
        raise RuntimeError(f'No ZIP downloaded for {scene_name}')
    extract_dir = scene_dir/'extracted'
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(extract_dir)
    safe_dirs = list(extract_dir.rglob('*.SAFE'))
    if not safe_dirs:
        raise RuntimeError(f'No SAFE found after extracting {scene_name}')
    return safe_dirs[0]


def cache_one_scene(row, product):
    scene_id = row.scene_id
    out_npz, out_meta = scene_cache_paths(scene_id)
    if out_npz.exists() and out_meta.exists() and not FORCE['sar_preprocessing']:
        return 'cached'

    temp_root = WORK_ROOT/'_TEMP_DOWNLOADS'
    temp_root.mkdir(exist_ok=True)
    safe = download_extract(product, temp_root)

    vv_tif, vv_ann, vv_cal = find_band_files(safe, 'vv')
    vh_tif, vh_ann, vh_cal = find_band_files(safe, 'vh')
    vv = calibrate_and_crop(vv_tif, vv_ann, vv_cal, CFG['lon_min'], CFG['lon_max'], CFG['lat_min'], CFG['lat_max'], CFG['crop_buffer_px'])
    vh = calibrate_and_crop(vh_tif, vh_ann, vh_cal, CFG['lon_min'], CFG['lon_max'], CFG['lat_min'], CFG['lat_max'], CFG['crop_buffer_px'])

    np.savez_compressed(out_npz,
        vv_db=vv['sigma0_db'].astype(np.float32),
        vh_db=vh['sigma0_db'].astype(np.float32),
        model_image=vv['model_image'].astype(np.uint8),
        geo_points=vv['geo_points'].astype(np.float64),
    )
    meta = {
        'scene_id': scene_id,
        'scene_name': row.scene_name,
        'scene_datetime': pd.to_datetime(row.start_time, utc=True).isoformat(),
        'date': row.date,
        'experiment_set': row.experiment_set,
        'vv_x_min': int(vv['x_min']), 'vv_y_min': int(vv['y_min']),
        'vv_x_max': int(vv['x_max']), 'vv_y_max': int(vv['y_max']),
        'vh_x_min': int(vh['x_min']), 'vh_y_min': int(vh['y_min']),
        'vh_x_max': int(vh['x_max']), 'vh_y_max': int(vh['y_max']),
        'pixel_spacing_m_approx': CFG['pixel_spacing_m_approx'],
        'calibration_note': 'sigma0 dB; apparent dimensions later use approximate GRD pixel spacing',
    }
    save_json(out_meta, meta)

    # Verify checkpoint before removing temporary source.
    test = np.load(out_npz, allow_pickle=False)
    assert test['vv_db'].ndim == 2 and test['model_image'].shape == test['vv_db'].shape

    if DELETE_TEMP_SAFE_AFTER_CACHE:
        shutil.rmtree(temp_root/row.scene_name, ignore_errors=True)
    return 'created'

In [ ]:
# STEP 10 - Build/reuse per-scene SAR caches
if RUN['sar_preprocessing']:
    # Resolve ASF product objects only for missing caches.
    missing_rows = []
    for row in allowed_catalog.itertuples(index=False):
        npz, meta = locate_scene_cache(row.scene_id)
        if FORCE['sar_preprocessing'] or npz is None:
            missing_rows.append(row)

    product_map = {}
    if missing_rows:
        names = [r.scene_name for r in missing_rows]
        products = asf.granule_search(names)
        product_map = {p.properties['sceneName']: p for p in products}
        unresolved = sorted(set(names) - set(product_map))
        if unresolved:
            raise RuntimeError(f'ASF could not resolve locked scenes: {unresolved}')

    errors = []
    for i, row in enumerate(allowed_catalog.itertuples(index=False), start=1):
        print(f'[{i}/{len(allowed_catalog)}] SAR cache {row.scene_id} {row.experiment_set}')
        try:
            npz, meta = locate_scene_cache(row.scene_id)
            if npz is not None and not FORCE['sar_preprocessing']:
                print('  SKIP - existing cache')
                continue
            status = cache_one_scene(row, product_map[row.scene_name])
            print(' ', status)
        except Exception as e:
            errors.append({'scene_id': row.scene_id, 'error': repr(e)})
            print('  ERROR:', repr(e))
            pd.DataFrame(errors).to_csv(DIRS['logs']/'sar_preprocessing_errors.csv', index=False)

    if errors:
        print('SAR preprocessing failures:', len(errors))
    stage_complete('SAR PREPROCESSING', scenes=len(allowed_catalog), checkpoint=DIRS['scene_cache'], errors=len(errors))

# Part D - Raw E0/E1 detection

The detector confidence threshold remains low (`0.25`) because this stage should preserve candidate recall. False-alarm rejection is deliberately deferred to later evidence stages.

In [ ]:
# STEP 11 - Tiled inference, global NMS and geolocation

def get_tile_positions(length, tile_size, stride):
    if length <= tile_size:
        return [0]
    pos = list(range(0, length-tile_size+1, stride))
    if pos[-1] != length-tile_size:
        pos.append(length-tile_size)
    return pos


def bbox_iou(a, b):
    ax1,ay1,ax2,ay2 = a
    bx1,by1,bx2,by2 = b
    ix1,iy1 = max(ax1,bx1), max(ay1,by1)
    ix2,iy2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    aa = max(0,ax2-ax1)*max(0,ay2-ay1)
    bb = max(0,bx2-bx1)*max(0,by2-by1)
    return inter/(aa+bb-inter+1e-9)


def numpy_nms(dets, iou_threshold=0.5):
    if not dets:
        return []
    arr = np.asarray(dets, dtype=np.float32)
    x1,y1,x2,y2,s = [arr[:,i] for i in range(5)]
    areas = np.maximum(0,x2-x1)*np.maximum(0,y2-y1)
    order = s.argsort()[::-1]
    keep=[]
    while len(order):
        i=order[0]; keep.append(i)
        if len(order)==1: break
        xx1=np.maximum(x1[i],x1[order[1:]]); yy1=np.maximum(y1[i],y1[order[1:]])
        xx2=np.minimum(x2[i],x2[order[1:]]); yy2=np.minimum(y2[i],y2[order[1:]])
        inter=np.maximum(0,xx2-xx1)*np.maximum(0,yy2-yy1)
        iou=inter/(areas[i]+areas[order[1:]]-inter+1e-9)
        order=order[np.where(iou<=iou_threshold)[0]+1]
    return arr[keep].tolist()


def run_tiled_inference(model, image):
    H,W=image.shape
    dets=[]
    for y in get_tile_positions(H, CFG['tile_size'], CFG['tile_stride']):
        for x in get_tile_positions(W, CFG['tile_size'], CFG['tile_stride']):
            tile = image[y:y+CFG['tile_size'], x:x+CFG['tile_size']]
            tile_bgr=cv2.cvtColor(tile, cv2.COLOR_GRAY2BGR)
            r=model.predict(source=tile_bgr, imgsz=CFG['tile_size'], conf=CFG['conf_threshold'],
                            iou=CFG['tile_nms_iou'], device=0 if torch.cuda.is_available() else 'cpu',
                            verbose=False, max_det=1000)[0]
            if r.boxes is None or len(r.boxes)==0:
                continue
            boxes=r.boxes.xyxy.cpu().numpy(); confs=r.boxes.conf.cpu().numpy()
            for (x1,y1,x2,y2),c in zip(boxes,confs):
                dets.append([x1+x,y1+y,x2+x,y2+y,float(c)])
    return numpy_nms(dets, CFG['global_nms_iou'])


def detections_to_df(dets, cache, scene_row, model_name):
    p2ll=build_pixel_to_lonlat(cache['geo_points'])
    rows=[]
    for i,(x1,y1,x2,y2,conf) in enumerate(dets):
        cx=(x1+x2)/2; cy=(y1+y2)/2
        gx1=x1+cache['vv_x_min']; gx2=x2+cache['vv_x_min']
        gy1=y1+cache['vv_y_min']; gy2=y2+cache['vv_y_min']
        gcx=cx+cache['vv_x_min']; gcy=cy+cache['vv_y_min']
        lon,lat=p2ll(gcx,gcy)
        rows.append({
            'scene_id':scene_row.scene_id,'scene_name':scene_row.scene_name,
            'scene_datetime':pd.to_datetime(scene_row.start_time,utc=True).isoformat(),
            'date':scene_row.date,'year':int(scene_row.date[:4]),'experiment_set':scene_row.experiment_set,
            'model':model_name,'detection_id':f'{scene_row.scene_id}_{model_name}_{i:04d}',
            'confidence':float(conf),
            'x1_crop':float(x1),'y1_crop':float(y1),'x2_crop':float(x2),'y2_crop':float(y2),
            'global_x1':float(gx1),'global_y1':float(gy1),'global_x2':float(gx2),'global_y2':float(gy2),
            'global_cx':float(gcx),'global_cy':float(gcy),
            'longitude':lon,'latitude':lat,
        })
    return pd.DataFrame(rows)

In [ ]:
# STEP 12 - Per-scene E0/E1 inference with checkpoints
STATUS_PATH = DIRS['raw_detections']/'SCENE_PROCESSING_STATUS.csv'
status_rows = []

if RUN['yolo_detection']:
    for row in allowed_catalog.itertuples(index=False):
        scene_status = {'scene_id':row.scene_id,'experiment_set':row.experiment_set,'last_updated':datetime.now(timezone.utc).isoformat()}
        try:
            cache = load_scene_cache(row.scene_id)
            for model_name, model in [('E0',model_e0),('E1',model_e1)]:
                out_csv = DIRS['raw_detections']/'per_scene'/f'{row.scene_id}_{model_name}.csv'
                old = read_path(Path('02_raw_detections/per_scene')/out_csv.name)
                if old.exists() and not FORCE['yolo_detection']:
                    df=pd.read_csv(old)
                    print(row.scene_id, model_name, 'SKIP', len(df))
                else:
                    dets=run_tiled_inference(model, cache['model_image'])
                    df=detections_to_df(dets, cache, row, model_name)
                    df.to_csv(out_csv,index=False)
                    print(row.scene_id, model_name, 'detections:', len(df))
                scene_status[f'{model_name}_status']='ok'
                scene_status[f'n_{model_name}']=len(df)
            scene_status['error']=''
        except Exception as e:
            scene_status['error']=repr(e)
            print('ERROR',row.scene_id,repr(e))
        status_rows.append(scene_status)
        pd.DataFrame(status_rows).to_csv(STATUS_PATH,index=False)

# Combine all currently allowed scene detections.
raw_frames=[]
for row in allowed_catalog.itertuples(index=False):
    for m in ['E0','E1']:
        p=read_path(Path('02_raw_detections/per_scene')/f'{row.scene_id}_{m}.csv')
        if p.exists(): raw_frames.append(pd.read_csv(p))
raw_detections=pd.concat(raw_frames,ignore_index=True) if raw_frames else pd.DataFrame()
save_table(raw_detections, DIRS['raw_detections']/'RAW_DETECTIONS_COMBINED')
stage_complete('YOLO CANDIDATE DETECTION', output_rows=len(raw_detections), scenes=raw_detections.scene_id.nunique() if len(raw_detections) else 0, checkpoint=DIRS['raw_detections'])

In [ ]:
# ============================================================
# STEP 12B - VISUAL E0 vs E1 COMPARISON
# Uses existing detections only. NO YOLO rerun.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

VIS_DIR = DIRS['raw_detections'] / 'E0_E1_visual_comparison'
VIS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def get_bbox_columns(df):
    """
    Automatically detect bounding-box column names.
    Supports common conventions used in our notebooks.
    """

    possible = [
        ('x1', 'y1', 'x2', 'y2'),
        ('x1_crop', 'y1_crop', 'x2_crop', 'y2_crop'),
        ('xmin', 'ymin', 'xmax', 'ymax'),
    ]

    for cols in possible:
        if all(c in df.columns for c in cols):
            return cols

    raise ValueError(
        f"Could not identify bbox columns.\n"
        f"Available columns:\n{df.columns.tolist()}"
    )


def box_iou(box1, box2):
    """
    IoU between two boxes:
    [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    intersection = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


def compare_detections(e0_df, e1_df, iou_threshold=0.30):
    """
    Match E0 and E1 detections using bounding-box IoU.

    Returns:
        matched_e0
        matched_e1
        e0_only
        e1_only
    """

    if len(e0_df) == 0:
        return set(), set(), set(), set(range(len(e1_df)))

    if len(e1_df) == 0:
        return set(), set(), set(range(len(e0_df))), set()

    e0_cols = get_bbox_columns(e0_df)
    e1_cols = get_bbox_columns(e1_df)

    e0_boxes = e0_df[list(e0_cols)].values.astype(float)
    e1_boxes = e1_df[list(e1_cols)].values.astype(float)

    candidate_matches = []

    for i, b0 in enumerate(e0_boxes):
        for j, b1 in enumerate(e1_boxes):

            iou = box_iou(b0, b1)

            if iou >= iou_threshold:
                candidate_matches.append((iou, i, j))

    # highest IoU matches first
    candidate_matches.sort(reverse=True)

    matched_e0 = set()
    matched_e1 = set()

    for iou, i, j in candidate_matches:

        if i not in matched_e0 and j not in matched_e1:
            matched_e0.add(i)
            matched_e1.add(j)

    e0_only = set(range(len(e0_df))) - matched_e0
    e1_only = set(range(len(e1_df))) - matched_e1

    return matched_e0, matched_e1, e0_only, e1_only


def prepare_image(img):
    """Make image suitable for matplotlib."""

    img = np.asarray(img)

    if img.ndim == 3 and img.shape[-1] == 1:
        img = img[..., 0]

    return img


def draw_boxes(ax, df, edgecolor, linewidth=1.3, label_conf=True):

    if len(df) == 0:
        return

    cols = get_bbox_columns(df)

    confidence_col = None

    for candidate in ['confidence', 'conf', 'score']:
        if candidate in df.columns:
            confidence_col = candidate
            break

    for _, r in df.iterrows():

        x1 = float(r[cols[0]])
        y1 = float(r[cols[1]])
        x2 = float(r[cols[2]])
        y2 = float(r[cols[3]])

        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=linewidth,
            edgecolor=edgecolor,
            facecolor='none'
        )

        ax.add_patch(rect)

        if label_conf and confidence_col is not None:

            conf = float(r[confidence_col])

            ax.text(
                x1,
                max(0, y1 - 2),
                f'{conf:.2f}',
                fontsize=6,
                color=edgecolor,
                bbox=dict(
                    facecolor='black',
                    alpha=0.45,
                    edgecolor='none',
                    pad=1
                )
            )


# ------------------------------------------------------------
# Generate visual comparison for every processed scene
# ------------------------------------------------------------

comparison_rows = []

for row in allowed_catalog.itertuples(index=False):

    scene_id = row.scene_id

    e0_path = read_path(
        Path('02_raw_detections/per_scene') / f'{scene_id}_E0.csv'
    )

    e1_path = read_path(
        Path('02_raw_detections/per_scene') / f'{scene_id}_E1.csv'
    )

    if not e0_path.exists() or not e1_path.exists():
        print('SKIP visualisation, missing CSV:', scene_id)
        continue

    e0 = pd.read_csv(e0_path)
    e1 = pd.read_csv(e1_path)

    # Load already prepared model image
    try:
        cache = load_scene_cache(scene_id)
        image = prepare_image(cache['model_image'])
    except Exception as exc:
        print('Could not load cached image:', scene_id, repr(exc))
        continue

    # Compare models
    matched_e0, matched_e1, e0_only, e1_only = compare_detections(
        e0,
        e1,
        iou_threshold=0.30
    )

    # --------------------------------------------------------
    # Save comparison statistics
    # --------------------------------------------------------

    comparison_rows.append({
        'scene_id': scene_id,
        'experiment_set': getattr(row, 'experiment_set', ''),
        'E0_total': len(e0),
        'E1_total': len(e1),
        'matched_E0_E1': len(matched_e0),
        'E0_only': len(e0_only),
        'E1_only': len(e1_only)
    })

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1, 3,
        figsize=(21, 7),
        constrained_layout=True
    )

    # ------------------
    # PANEL 1 : E0
    # ------------------

    axes[0].imshow(image, cmap='gray')
    draw_boxes(
        axes[0],
        e0,
        edgecolor='red'
    )

    axes[0].set_title(
        f'E0 baseline\n{len(e0)} detections',
        fontsize=12
    )

    axes[0].axis('off')

    # ------------------
    # PANEL 2 : E1
    # ------------------

    axes[1].imshow(image, cmap='gray')
    draw_boxes(
        axes[1],
        e1,
        edgecolor='blue'
    )

    axes[1].set_title(
        f'E1 SensorMix\n{len(e1)} detections',
        fontsize=12
    )

    axes[1].axis('off')

    # ------------------
    # PANEL 3 : DIFFERENCE
    # ------------------

    axes[2].imshow(image, cmap='gray')

    # detections common to both models
    if matched_e0:
        draw_boxes(
            axes[2],
            e0.iloc[sorted(matched_e0)],
            edgecolor='lime',
            linewidth=1.5,
            label_conf=False
        )

    # E0 only
    if e0_only:
        draw_boxes(
            axes[2],
            e0.iloc[sorted(e0_only)],
            edgecolor='red',
            linewidth=1.5,
            label_conf=False
        )

    # E1 only
    if e1_only:
        draw_boxes(
            axes[2],
            e1.iloc[sorted(e1_only)],
            edgecolor='cyan',
            linewidth=1.5,
            label_conf=False
        )

    axes[2].set_title(
        'E0 vs E1\n'
        f'Common={len(matched_e0)} | '
        f'E0 only={len(e0_only)} | '
        f'E1 only={len(e1_only)}',
        fontsize=12
    )

    axes[2].axis('off')

    # manual legend
    legend_handles = [
        patches.Patch(
            facecolor='none',
            edgecolor='lime',
            label='Detected by E0 + E1'
        ),
        patches.Patch(
            facecolor='none',
            edgecolor='red',
            label='E0 only'
        ),
        patches.Patch(
            facecolor='none',
            edgecolor='cyan',
            label='E1 only'
        )
    ]

    axes[2].legend(
        handles=legend_handles,
        loc='lower right',
        fontsize=8
    )

    fig.suptitle(
        f'{scene_id} | {getattr(row, "experiment_set", "")}',
        fontsize=14
    )

    output_file = VIS_DIR / f'{scene_id}_E0_vs_E1.jpg'

    plt.savefig(
        output_file,
        dpi=160,
        bbox_inches='tight'
    )

    plt.close(fig)

    print(
        f'{scene_id}: '
        f'E0={len(e0)} | '
        f'E1={len(e1)} | '
        f'common={len(matched_e0)} | '
        f'E0-only={len(e0_only)} | '
        f'E1-only={len(e1_only)}'
    )


# ------------------------------------------------------------
# Save model comparison table
# ------------------------------------------------------------

comparison_df = pd.DataFrame(comparison_rows)

comparison_csv = VIS_DIR / 'E0_E1_SCENE_COMPARISON.csv'
comparison_df.to_csv(comparison_csv, index=False)

print('\n' + '=' * 90)
print('E0 / E1 VISUAL COMPARISON COMPLETE')
print('=' * 90)

display(comparison_df)

print('\nImages saved to:')
print(VIS_DIR)

print('\nComparison table:')
print(comparison_csv)

# Part E - E0/E1 one-to-one candidate fusion

Two model outputs are merged into one canonical physical target. E0/E1 agreement is treated as **model-consensus evidence**, not as independent ground truth.

In [ ]:
# STEP 13 - Fusion helpers

def fuse_scene_models(scene_df):
    e0=scene_df[scene_df.model=='E0'].reset_index(drop=True)
    e1=scene_df[scene_df.model=='E1'].reset_index(drop=True)
    pairs=[]; used0=set(); used1=set()
    if len(e0) and len(e1):
        cost=np.full((len(e0),len(e1)),1e6,dtype=float)
        for i,a in e0.iterrows():
            for j,b in e1.iterrows():
                ca=np.array([a.global_cx,a.global_cy]); cb=np.array([b.global_cx,b.global_cy])
                dist=float(np.linalg.norm(ca-cb))
                iou=bbox_iou([a.global_x1,a.global_y1,a.global_x2,a.global_y2],[b.global_x1,b.global_y1,b.global_x2,b.global_y2])
                if dist<=CFG['fusion_max_center_px'] or iou>=CFG['fusion_min_iou']:
                    cost[i,j]=(dist/CFG['fusion_max_center_px']) + (1.0-iou)
        rr,cc=linear_sum_assignment(cost)
        for i,j in zip(rr,cc):
            if cost[i,j] < 1e5:
                pairs.append((i,j)); used0.add(i); used1.add(j)

    rows=[]
    def canonical(a=None,b=None):
        src=[x for x in [a,b] if x is not None]
        weights=np.array([float(x.confidence) for x in src]); weights=weights/(weights.sum()+1e-9)
        def wav(col): return float(sum(w*float(x[col]) for w,x in zip(weights,src)))
        base=src[0]
        return {
            'scene_id':base.scene_id,'scene_name':base.scene_name,'scene_datetime':base.scene_datetime,
            'date':base.date,'year':int(base.year),'experiment_set':base.experiment_set,
            'E0_detected':a is not None,'E1_detected':b is not None,'model_consensus':(a is not None and b is not None),
            'E0_confidence':float(a.confidence) if a is not None else np.nan,
            'E1_confidence':float(b.confidence) if b is not None else np.nan,
            'confidence_max':float(max(x.confidence for x in src)),
            'confidence_mean':float(np.mean([x.confidence for x in src])),
            'global_x1':wav('global_x1'),'global_y1':wav('global_y1'),'global_x2':wav('global_x2'),'global_y2':wav('global_y2'),
            'global_cx':wav('global_cx'),'global_cy':wav('global_cy'),
            'longitude':wav('longitude'),'latitude':wav('latitude'),
        }
    for i,j in pairs: rows.append(canonical(e0.iloc[i],e1.iloc[j]))
    for i in range(len(e0)):
        if i not in used0: rows.append(canonical(e0.iloc[i],None))
    for j in range(len(e1)):
        if j not in used1: rows.append(canonical(None,e1.iloc[j]))
    out=pd.DataFrame(rows)
    if len(out):
        out=out.sort_values(['latitude','longitude','confidence_max'],ascending=[True,True,False]).reset_index(drop=True)
        out['candidate_id']=[f"{out.scene_id.iloc[0]}_C{i:04d}" for i in range(1,len(out)+1)]
    return out

In [ ]:
# STEP 14 - Run/reuse candidate fusion, incrementally adding newly processed scenes
fused_old = None if FORCE['candidate_fusion'] else load_table('03_candidate_fusion/FUSED_SAR_CANDIDATES')
if fused_old is None:
    fused_old = pd.DataFrame()

raw_scene_ids=set(raw_detections.scene_id.unique()) if len(raw_detections) else set()
old_scene_ids=set(fused_old.scene_id.unique()) if len(fused_old) else set()
missing_scene_ids=sorted(raw_scene_ids-old_scene_ids)

if missing_scene_ids or FORCE['candidate_fusion'] or len(fused_old)==0:
    if not RUN['candidate_fusion']:
        raise RuntimeError('Fusion checkpoint is incomplete and fusion stage disabled.')
    frames=[] if FORCE['candidate_fusion'] else ([fused_old] if len(fused_old) else [])
    target_ids=sorted(raw_scene_ids) if FORCE['candidate_fusion'] else missing_scene_ids
    for sid in target_ids:
        sdf=raw_detections[raw_detections.scene_id==sid].copy()
        frames.append(fuse_scene_models(sdf))
        print('Fused newly available scene:',sid)
    fused=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
    fused=fused.drop_duplicates('candidate_id',keep='last') if len(fused) else fused
    save_table(fused, DIRS['fusion']/'FUSED_SAR_CANDIDATES')
else:
    fused=fused_old.copy()

# Restrict the in-memory table to scenes allowed for this run, while the checkpoint may contain more.
fused=fused[fused.scene_id.isin(raw_scene_ids)].copy() if len(fused) else fused
assert fused['candidate_id'].is_unique if len(fused) else True
stage_complete('E0/E1 CANDIDATE FUSION', input_rows=len(raw_detections), output_rows=len(fused), scenes=fused.scene_id.nunique() if len(fused) else 0, checkpoint=DIRS['fusion'])

# Part F - Land, island and coastal context

The old V3 notebook only defined coastal thresholds but never implemented them. V4 uses candidate-box overlap, target-pixel overlap later, and distance to coastline. Detections within 300 m of coast are **not automatically deleted** because genuine ships can be berthed or near terminals.

Shoreline source: GSHHG v2.3.7 full-resolution Level-1 polygons. The code first looks for an attached `GSHHS_f_L1.shp` under `/kaggle/input`; otherwise it downloads the official SOEST package once.

In [ ]:
# STEP 15 - Load high-detail GSHHG shoreline polygons and clip to AOI
GSHHG_URLS = [
    'https://ftp.soest.hawaii.edu/gshhg/gshhg-shp-2.3.7.zip',
    'http://www.soest.hawaii.edu/pwessel/gshhg/gshhg-shp-2.3.7.zip',
]


def find_attached_gshhg():
    files=list(KAGGLE_INPUT.rglob('GSHHS_f_L1.shp')) if KAGGLE_INPUT.exists() else []
    return files[0] if files else None


def ensure_gshhg_shapefile():
    attached=find_attached_gshhg()
    if attached:
        print('Using attached GSHHG:',attached)
        return attached
    local=DIRS['spatial']/'gshhg_shp_2.3.7'
    shp=local/'GSHHS_shp'/'f'/'GSHHS_f_L1.shp'
    old=read_path(Path('04_spatial_context/gshhg_shp_2.3.7/GSHHS_shp/f/GSHHS_f_L1.shp'))
    if old.exists() and not FORCE['spatial_context']:
        return old
    local.mkdir(parents=True,exist_ok=True)
    zpath=DIRS['spatial']/'gshhg-shp-2.3.7.zip'
    last_err=None
    for url in GSHHG_URLS:
        try:
            print('Downloading GSHHG:',url)
            with requests.get(url,stream=True,timeout=180) as r:
                r.raise_for_status()
                with open(zpath,'wb') as f:
                    for chunk in r.iter_content(1024*1024):
                        if chunk: f.write(chunk)
            with zipfile.ZipFile(zpath) as z: z.extractall(local)
            if shp.exists(): return shp
        except Exception as e:
            last_err=e; print('GSHHG source failed:',repr(e))
    raise RuntimeError(f'Could not obtain GSHHG full-resolution shoreline. Attach GSHHS_f_L1.shp as a Kaggle dataset. Last error: {last_err}')

GSHHG_SHP=ensure_gshhg_shapefile()
shore=gpd.read_file(GSHHG_SHP, bbox=(CFG['lon_min']-0.05,CFG['lat_min']-0.05,CFG['lon_max']+0.05,CFG['lat_max']+0.05))
if shore.crs is None: shore=shore.set_crs(4326)
shore=shore.to_crs(4326)
aoi_geom=box(CFG['lon_min']-0.03,CFG['lat_min']-0.03,CFG['lon_max']+0.03,CFG['lat_max']+0.03)
shore=shore[shore.intersects(aoi_geom)].copy()
land_union_wgs84=unary_union(shore.geometry.tolist())

transform_to_utm=Transformer.from_crs(4326,CFG['utm_epsg'],always_xy=True).transform
land_union_utm=shapely_transform(transform_to_utm,land_union_wgs84)
coast_boundary_utm=land_union_utm.boundary
print('GSHHG polygons intersecting AOI:',len(shore))

In [ ]:
# =============================================================================
# STEP 16 - FAST + RESUME-SAFE SPATIAL CONTEXT
# Land / island / coastline characterization
#
# Improvements:
#   1. Load each SAR scene cache only ONCE
#   2. Build pixel→lon/lat converter only ONCE per scene
#   3. Skip expensive intersection when bbox clearly does not touch land
#   4. Save checkpoint AFTER EVERY SCENE
#   5. Resume only missing candidate IDs
#   6. tqdm progress bars show elapsed time + ETA
# =============================================================================

import time
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from shapely.geometry import Point, Polygon
from shapely.ops import transform as shapely_transform
from shapely.prepared import prep


# =============================================================================
# 1. PREPARE LAND GEOMETRY ONCE
# =============================================================================

print("=" * 100)
print("STEP 16 - LAND / ISLAND / COAST SPATIAL CONTEXT")
print("=" * 100)

step_start = time.perf_counter()

# Prepared geometry speeds up repeated intersects() tests.
land_prepared_utm = prep(land_union_utm)

print("Prepared full-resolution GSHHG land geometry.")
print("Total fused candidates:", len(fused))
print("Total scenes:", fused["scene_id"].nunique() if len(fused) else 0)


# =============================================================================
# 2. COAST ZONE
# =============================================================================

def coast_zone(distance_m, center_on_land, bbox_overlap):

    # Any actual land overlap is retained as strong spatial evidence.
    # Nothing is deleted here.
    if center_on_land or bbox_overlap > 0:
        return "LAND_OR_INTERSECTING"

    if distance_m <= CFG["coast_zone_1_m"]:
        return "0_TO_300_M"

    if distance_m <= CFG["coast_zone_2_m"]:
        return "300_TO_1000_M"

    return "GT_1000_M"


# =============================================================================
# 3. BUILD BBOX POLYGON
# =============================================================================

def candidate_bbox_polygon_fast(row, p2ll):
    """
    Convert the four corners of one YOLO candidate box
    from global image pixels into lon/lat polygon.

    p2ll is already constructed ONCE for the scene.
    """

    corners = [
        p2ll(row.global_x1, row.global_y1),
        p2ll(row.global_x2, row.global_y1),
        p2ll(row.global_x2, row.global_y2),
        p2ll(row.global_x1, row.global_y2),
    ]

    return Polygon(corners)


# =============================================================================
# 4. PROCESS ONE CANDIDATE
# =============================================================================

def spatial_features_one_candidate(row, p2ll):

    # -------------------------------------------------------------------------
    # A. Candidate bounding box: pixels → lon/lat
    # -------------------------------------------------------------------------

    bbox_wgs = candidate_bbox_polygon_fast(
        row,
        p2ll
    )

    # -------------------------------------------------------------------------
    # B. lon/lat → UTM metres
    # -------------------------------------------------------------------------

    bbox_utm = shapely_transform(
        transform_to_utm,
        bbox_wgs
    )

    bbox_area = max(
        float(bbox_utm.area),
        1e-9
    )

    # -------------------------------------------------------------------------
    # C. Land overlap
    #
    # First perform cheap intersects test.
    # Only perform expensive exact intersection-area calculation
    # when the bbox actually intersects land.
    # -------------------------------------------------------------------------

    if land_prepared_utm.intersects(bbox_utm):

        overlap_area = float(
            bbox_utm
            .intersection(land_union_utm)
            .area
        )

        bbox_overlap = overlap_area / bbox_area

    else:

        bbox_overlap = 0.0


    bbox_overlap = float(
        np.clip(
            bbox_overlap,
            0.0,
            1.0
        )
    )

    # -------------------------------------------------------------------------
    # D. Candidate centre
    # -------------------------------------------------------------------------

    point_wgs = Point(
        float(row.longitude),
        float(row.latitude)
    )

    point_utm = shapely_transform(
        transform_to_utm,
        point_wgs
    )

    # covers() includes interior + boundary
    center_on_land = bool(
        land_union_utm.covers(point_utm)
    )

    # -------------------------------------------------------------------------
    # E. Exact distance from centre to GSHHG coastline
    # -------------------------------------------------------------------------

    distance_to_coast_m = float(
        point_utm.distance(
            coast_boundary_utm
        )
    )

    # -------------------------------------------------------------------------
    # F. Coast category
    # -------------------------------------------------------------------------

    zone = coast_zone(
        distance_to_coast_m,
        center_on_land,
        bbox_overlap
    )

    return {
        "candidate_id": row.candidate_id,

        "center_on_land":
            center_on_land,

        "bbox_land_overlap_fraction":
            bbox_overlap,

        "distance_to_coast_m":
            distance_to_coast_m,

        "coast_zone":
            zone,
    }


# =============================================================================
# 5. LOAD EXISTING CHECKPOINT
# =============================================================================

if FORCE["spatial_context"]:

    print("\nFORCE spatial_context = True")
    print("Existing spatial results will be recalculated.")

    spatial = pd.DataFrame()

else:

    spatial = load_table(
        "04_spatial_context/SPATIAL_CONTEXT"
    )

    if spatial is None:
        spatial = pd.DataFrame()


if len(spatial):

    spatial["candidate_id"] = (
        spatial["candidate_id"]
        .astype(str)
    )

    completed_ids = set(
        spatial["candidate_id"]
    )

else:

    completed_ids = set()


print("\nAlready completed candidates:", len(completed_ids))


# =============================================================================
# 6. FIND ONLY CANDIDATES THAT STILL NEED PROCESSING
# =============================================================================

fused_work = fused.copy()

fused_work["_candidate_id_str"] = (
    fused_work["candidate_id"]
    .astype(str)
)


if FORCE["spatial_context"]:

    need = fused_work.copy()

else:

    need = fused_work[
        ~fused_work["_candidate_id_str"]
        .isin(completed_ids)
    ].copy()


print("Candidates still requiring spatial processing:", len(need))


# =============================================================================
# 7. NOTHING TO DO?
# =============================================================================

if len(need) == 0:

    print("\nAll candidates already have spatial-context results.")
    print("No geospatial calculations required.")


# =============================================================================
# 8. PROCESS SCENE BY SCENE
# =============================================================================

else:

    scene_groups = list(
        need.groupby(
            "scene_id",
            sort=False
        )
    )

    print("\nScenes requiring processing:", len(scene_groups))
    print()


    # Outer bar:
    # scene progress + overall ETA
    scene_bar = tqdm(
        scene_groups,
        total=len(scene_groups),
        desc="Spatial scenes",
        unit="scene",
        dynamic_ncols=True
    )


    for scene_number, (scene_id, scene_df) in enumerate(
        scene_bar,
        start=1
    ):

        scene_start = time.perf_counter()

        scene_bar.set_postfix_str(
            f"{scene_id} | {len(scene_df)} candidates"
        )

        print("\n" + "-" * 100)
        print(
            f"SCENE {scene_number}/{len(scene_groups)} : {scene_id}"
        )
        print(
            f"Candidates to process: {len(scene_df)}"
        )


        # =====================================================================
        # LOAD SAR CACHE ONCE FOR THIS SCENE
        # =====================================================================

        try:

            cache = load_scene_cache(
                scene_id
            )

            # Build pixel → lon/lat converter ONCE
            p2ll = build_pixel_to_lonlat(
                cache["geo_points"]
            )

        except Exception as e:

            print(
                "ERROR loading scene cache:",
                scene_id,
                repr(e)
            )

            continue


        # =====================================================================
        # PROCESS ALL CANDIDATES IN THIS SCENE
        # =====================================================================

        scene_rows = []
        candidate_errors = 0


        candidate_bar = tqdm(
            scene_df.itertuples(index=False),
            total=len(scene_df),
            desc=f"  {scene_id}",
            unit="candidate",
            leave=False,
            dynamic_ncols=True
        )


        for row in candidate_bar:

            try:

                result = spatial_features_one_candidate(
                    row,
                    p2ll
                )

                scene_rows.append(
                    result
                )

            except Exception as e:

                candidate_errors += 1

                # Do not add a fake spatial record.
                # Missing candidate will automatically retry
                # on the next notebook run.

                if candidate_errors <= 5:

                    print(
                        "\nCandidate spatial error:",
                        row.candidate_id,
                        repr(e)
                    )


        # =====================================================================
        # SAVE THIS SCENE IMMEDIATELY
        # =====================================================================

        if scene_rows:

            scene_spatial = pd.DataFrame(
                scene_rows
            )

            if len(spatial):

                spatial = pd.concat(
                    [
                        spatial,
                        scene_spatial
                    ],
                    ignore_index=True
                )

            else:

                spatial = scene_spatial.copy()


            spatial["candidate_id"] = (
                spatial["candidate_id"]
                .astype(str)
            )


            spatial = (
                spatial
                .drop_duplicates(
                    subset="candidate_id",
                    keep="last"
                )
                .reset_index(drop=True)
            )


            # -------------------------------------------------------------
            # CRITICAL:
            # checkpoint is written AFTER EVERY SCENE
            # -------------------------------------------------------------

            save_table(
                spatial,
                DIRS["spatial"] / "SPATIAL_CONTEXT"
            )


        # =====================================================================
        # SCENE TIMING REPORT
        # =====================================================================

        scene_elapsed = (
            time.perf_counter()
            - scene_start
        )


        if scene_elapsed > 0:

            speed = (
                len(scene_rows)
                / scene_elapsed
            )

        else:

            speed = np.nan


        print(
            f"Completed: {len(scene_rows)}/{len(scene_df)}"
        )

        print(
            f"Errors   : {candidate_errors}"
        )

        print(
            f"Time     : {scene_elapsed:.1f} sec"
        )

        print(
            f"Speed    : {speed:.2f} candidates/sec"
        )

        print(
            "Checkpoint saved:",
            DIRS["spatial"] / "SPATIAL_CONTEXT"
        )


        # Free scene objects before next scene
        del cache
        del p2ll


# =============================================================================
# 9. RELOAD FINAL SAVED SPATIAL TABLE
# =============================================================================

spatial_final = load_table(
    "04_spatial_context/SPATIAL_CONTEXT"
)


if spatial_final is not None:

    spatial = spatial_final.copy()


# Only retain candidates belonging to current fused catalogue
if len(spatial):

    valid_ids = set(
        fused["candidate_id"]
        .astype(str)
    )

    spatial["candidate_id"] = (
        spatial["candidate_id"]
        .astype(str)
    )

    spatial = spatial[
        spatial["candidate_id"]
        .isin(valid_ids)
    ].copy()


# =============================================================================
# 10. FINAL SUMMARY
# =============================================================================

total_elapsed = (
    time.perf_counter()
    - step_start
)


print("\n" + "=" * 100)
print("SPATIAL CONTEXT COMPLETE")
print("=" * 100)

print(
    f"Input fused candidates : {len(fused)}"
)

print(
    f"Spatial rows available : {len(spatial)}"
)

print(
    f"Total elapsed time      : {total_elapsed / 60:.2f} minutes"
)


if len(spatial):

    print("\nCoast-zone counts:")

    print(
        spatial["coast_zone"]
        .value_counts(
            dropna=False
        )
    )


    print("\nCoast-zone percentages:")

    print(
        (
            spatial["coast_zone"]
            .value_counts(
                normalize=True,
                dropna=False
            )
            .mul(100)
            .round(2)
        )
    )


    print(
        "\nCandidates with centre on land:",
        int(spatial["center_on_land"].sum())
    )


    print(
        "Candidates with any bbox-land overlap:",
        int(
            (
                spatial["bbox_land_overlap_fraction"]
                > 0
            ).sum()
        )
    )


print("\nCheckpoint:")
print(
    DIRS["spatial"] / "SPATIAL_CONTEXT"
)


stage_complete(
    "LAND / ISLAND / COAST CONTEXT",
    input_rows=len(fused),
    output_rows=len(spatial),
    checkpoint=DIRS["spatial"]
)

In [ ]:
# ============================================================
# STEP 16B - SPIN-OFF PLOTS FOR LAND / COAST ANALYSIS
# Run AFTER Step 16
# Does NOT change/filter any detections
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ------------------------------------------------------------
# 1. Merge fused candidates with spatial context
# ------------------------------------------------------------

plot_df = fused.merge(
    spatial,
    on='candidate_id',
    how='left'
)


# ------------------------------------------------------------
# 2. Identify E0-only / E1-only / common candidates
# ------------------------------------------------------------

def get_detector_source(r):

    if bool(r['E0_detected']) and bool(r['E1_detected']):
        return 'E0+E1'

    elif bool(r['E0_detected']):
        return 'E0_only'

    elif bool(r['E1_detected']):
        return 'E1_only'

    else:
        return 'UNKNOWN'


plot_df['detector_source'] = plot_df.apply(
    get_detector_source,
    axis=1
)


# ------------------------------------------------------------
# 3. DEVELOPMENT scenes only
# ------------------------------------------------------------

print("Available experiment sets:")
print(plot_df['experiment_set'].value_counts(dropna=False))


dev_plot = plot_df[
    plot_df['experiment_set']
    .astype(str)
    .str.lower()
    .isin(['development', 'dev'])
].copy()


print("\nDevelopment candidates:", len(dev_plot))

print("\nDetector-source counts:")
print(dev_plot['detector_source'].value_counts())


# ------------------------------------------------------------
# 4. Output folder
# ------------------------------------------------------------

PLOT_DIR = DIRS['spatial'] / 'diagnostic_plots'
PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nPlots will be saved to:")
print(PLOT_DIR)


# ============================================================
# SUMMARY TABLE 1
# Coast-zone counts
# ============================================================

zone_order = [
    'LAND_OR_INTERSECTING',
    '0_TO_300_M',
    '300_TO_1000_M',
    'GT_1000_M'
]

source_order = [
    'E0_only',
    'E1_only',
    'E0+E1'
]


zone_table = pd.crosstab(
    dev_plot['detector_source'],
    dev_plot['coast_zone']
)


zone_table = zone_table.reindex(
    index=[
        x for x in source_order
        if x in zone_table.index
    ],
    columns=[
        x for x in zone_order
        if x in zone_table.columns
    ],
    fill_value=0
)


print("\n" + "="*90)
print("COAST ZONE COUNTS")
print("="*90)

display(zone_table)


zone_table.to_csv(
    PLOT_DIR / 'COAST_ZONE_COUNTS.csv'
)


# ============================================================
# SUMMARY TABLE 2
# Coast-zone percentages
# ============================================================

zone_pct = (
    zone_table
    .div(zone_table.sum(axis=1), axis=0)
    * 100
)


print("\n" + "="*90)
print("COAST ZONE PERCENTAGES")
print("="*90)

display(zone_pct.round(2))


zone_pct.to_csv(
    PLOT_DIR / 'COAST_ZONE_PERCENTAGES.csv'
)


# ============================================================
# PLOT 1
# Absolute number of candidates in each coast zone
# ============================================================

ax = zone_table.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6)
)

ax.set_title(
    'Development Set: Spatial Context of E0 and E1 Candidates'
)

ax.set_xlabel(
    'Detection source'
)

ax.set_ylabel(
    'Number of fused candidates'
)

ax.legend(
    title='Coastal context',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.xticks(
    rotation=0
)

plt.tight_layout()


out1 = (
    PLOT_DIR /
    '01_COAST_ZONE_COUNTS.png'
)

plt.savefig(
    out1,
    dpi=250,
    bbox_inches='tight'
)

plt.show()

print("Saved:", out1)


# ============================================================
# PLOT 2
# Percentage distribution
# Better for comparing E0-only vs E1-only fairly
# ============================================================

ax = zone_pct.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6)
)

ax.set_title(
    'Development Set: Percentage of Candidates by Coastal Context'
)

ax.set_xlabel(
    'Detection source'
)

ax.set_ylabel(
    'Candidates (%)'
)

ax.set_ylim(
    0,
    100
)

ax.legend(
    title='Coastal context',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.xticks(
    rotation=0
)

plt.tight_layout()


out2 = (
    PLOT_DIR /
    '02_COAST_ZONE_PERCENTAGES.png'
)

plt.savefig(
    out2,
    dpi=250,
    bbox_inches='tight'
)

plt.show()

print("Saved:", out2)


# ============================================================
# PLOT 3
# Land/intersection percentage only
# This directly tests your visual observation
# ============================================================

land_rate = (
    dev_plot
    .assign(
        land_flag=
        dev_plot['coast_zone']
        .eq('LAND_OR_INTERSECTING')
    )
    .groupby('detector_source')['land_flag']
    .mean()
    .mul(100)
)


land_rate = land_rate.reindex(
    [
        x for x in source_order
        if x in land_rate.index
    ]
)


print("\nLand/intersection percentage:")

display(
    land_rate
    .round(2)
    .rename('land_or_intersecting_percent')
    .to_frame()
)


ax = land_rate.plot(
    kind='bar',
    figsize=(8, 5)
)

ax.set_title(
    'Development Set: Land-Associated Detection Rate'
)

ax.set_xlabel(
    'Detection source'
)

ax.set_ylabel(
    'Land or intersecting (%)'
)

ax.set_ylim(
    0,
    max(100, land_rate.max() * 1.15)
)

plt.xticks(
    rotation=0
)

plt.tight_layout()


out3 = (
    PLOT_DIR /
    '03_LAND_INTERSECTION_RATE.png'
)

plt.savefig(
    out3,
    dpi=250,
    bbox_inches='tight'
)

plt.show()

print("Saved:", out3)


# ============================================================
# PLOT 4
# Distance from coastline for OFFSHORE candidates only
# ============================================================

offshore = dev_plot[
    dev_plot['coast_zone']
    != 'LAND_OR_INTERSECTING'
].copy()


offshore = offshore[
    np.isfinite(
        offshore['distance_to_coast_m']
    )
]


groups = []
labels = []


for source in source_order:

    values = offshore.loc[
        offshore['detector_source'] == source,
        'distance_to_coast_m'
    ].values

    if len(values) > 0:

        groups.append(values)

        labels.append(source)


if groups:

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.boxplot(
        groups,
        tick_labels=labels,
        showfliers=False
    )

    ax.set_yscale(
        'log'
    )

    ax.set_title(
        'Development Set: Offshore Distance to Coast'
    )

    ax.set_xlabel(
        'Detection source'
    )

    ax.set_ylabel(
        'Distance to coastline (m, log scale)'
    )

    plt.tight_layout()


    out4 = (
        PLOT_DIR /
        '04_OFFSHORE_DISTANCE_TO_COAST.png'
    )

    plt.savefig(
        out4,
        dpi=250,
        bbox_inches='tight'
    )

    plt.show()

    print("Saved:", out4)


print("\n" + "="*90)
print("SPATIAL DIAGNOSTIC PLOTTING COMPLETE")
print("="*90)

print("\nSaved folder:")
print(PLOT_DIR)


# Part G - SAR target properties

Target properties are calculated from the calibrated VV/VH scene cache.

V4.1 segments a locally bright connected component that must correspond to the detector region, rather than accepting an unrelated nearby bright shoreline component. Geometry is scaled using **local ground spacing estimated from Sentinel-1 geolocation tie points** for each candidate, with the configured ~10 m spacing used only as a fallback.

The outputs remain **apparent SAR-signature properties**, not guaranteed true hull dimensions. Their usefulness is checked later using AIS-reported vessel length where available.


In [ ]:
# =============================================================================
# STEP 17 - ROBUST + OPTIMIZED SAR FEATURE EXTRACTION
#
# Scientific logic preserved:
#   - robust median/MAD background
#   - adaptive target threshold
#   - morphology
#   - connected-component target selection
#   - SAR apparent geometry
#   - VV/VH target/background statistics
#   - target land-overlap fraction
#
# Main speed changes:
#   1. Scene cache is PASSED IN, not loaded for every candidate
#   2. pixel→lon/lat converter is PASSED IN, not rebuilt every candidate
#   3. UTM transformation of component samples is vectorized
#   4. land point tests use Shapely vectorized operations where available
# =============================================================================

import math
import numpy as np
import cv2
import shapely

from shapely.geometry import Point
from shapely.prepared import prep


# =============================================================================
# 1. PREPARE LAND GEOMETRY ONCE
# =============================================================================

prepared_land_union_utm = prep(land_union_utm)


# =============================================================================
# 2. ROBUST STATISTICS
# =============================================================================

def robust_stats(x):

    x = np.asarray(
        x,
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    med = float(
        np.median(x)
    )

    mad = float(
        np.median(
            np.abs(x - med)
        )
    )

    sig = 1.4826 * mad

    return med, mad, sig


# =============================================================================
# 3. GLOBAL COORDINATES → ARRAY CROP
# =============================================================================

def crop_global(
    arr,
    x_min,
    y_min,
    gx1,
    gy1,
    gx2,
    gy2
):

    x1 = max(
        0,
        int(
            math.floor(
                gx1 - x_min
            )
        )
    )

    y1 = max(
        0,
        int(
            math.floor(
                gy1 - y_min
            )
        )
    )

    x2 = min(
        arr.shape[1],
        int(
            math.ceil(
                gx2 - x_min
            )
        )
    )

    y2 = min(
        arr.shape[0],
        int(
            math.ceil(
                gy2 - y_min
            )
        )
    )

    return (
        arr[y1:y2, x1:x2],
        (x1, y1, x2, y2)
    )


# =============================================================================
# 4. SEGMENT SAR SIGNATURE
# =============================================================================

def segment_signature(
    vv,
    bbox_local,
    margin=12
):

    x1, y1, x2, y2 = [
        int(round(v))
        for v in bbox_local
    ]

    H, W = vv.shape

    wx1 = max(
        0,
        x1 - margin
    )

    wy1 = max(
        0,
        y1 - margin
    )

    wx2 = min(
        W,
        x2 + margin
    )

    wy2 = min(
        H,
        y2 + margin
    )


    win = vv[
        wy1:wy2,
        wx1:wx2
    ]


    # -------------------------------------------------------------------------
    # Window QA
    # -------------------------------------------------------------------------

    if (
        win.size < 20
        or
        np.isfinite(win).sum() < 20
    ):

        return (
            None,
            {
                "quality":
                    "FAILED_SMALL_WINDOW"
            }
        )


    # -------------------------------------------------------------------------
    # Background = local window excluding detector bbox
    # -------------------------------------------------------------------------

    mask_bg = np.ones(
        win.shape,
        dtype=bool
    )


    bx1 = max(
        0,
        x1 - wx1
    )

    by1 = max(
        0,
        y1 - wy1
    )

    bx2 = min(
        win.shape[1],
        x2 - wx1
    )

    by2 = min(
        win.shape[0],
        y2 - wy1
    )


    mask_bg[
        by1:by2,
        bx1:bx2
    ] = False


    bg = win[
        mask_bg
    ]


    bg_med, bg_mad, bg_sig = robust_stats(
        bg
    )


    if not np.isfinite(
        bg_med
    ):

        return (
            None,
            {
                "quality":
                    "FAILED_BACKGROUND"
            }
        )


    # -------------------------------------------------------------------------
    # Adaptive SAR target threshold
    # -------------------------------------------------------------------------

    thr = (
        bg_med
        +
        max(
            2.5 * bg_sig,
            2.0
        )
    )


    binary = (
        np.isfinite(win)
        &
        (win >= thr)
    ).astype(
        np.uint8
    )


    # -------------------------------------------------------------------------
    # Morphological cleanup
    # -------------------------------------------------------------------------

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        np.ones(
            (2, 2),
            np.uint8
        )
    )


    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        np.ones(
            (3, 3),
            np.uint8
        )
    )


    # -------------------------------------------------------------------------
    # Connected components
    # -------------------------------------------------------------------------

    n, labels, stats, cent = (
        cv2.connectedComponentsWithStats(
            binary,
            8
        )
    )


    if n <= 1:

        return (
            None,
            {
                "quality":
                    "NO_COMPONENT",

                "bg_med":
                    bg_med,

                "bg_sig":
                    bg_sig,

                "threshold":
                    thr
            }
        )


    # -------------------------------------------------------------------------
    # Prefer component closest to YOLO detection centre
    # -------------------------------------------------------------------------

    target_center = np.array(
        [
            (bx1 + bx2) / 2,
            (by1 + by2) / 2
        ]
    )


    candidates = []


    for lab in range(
        1,
        n
    ):

        area = stats[
            lab,
            cv2.CC_STAT_AREA
        ]


        if area < 2:
            continue


        d = float(
            np.linalg.norm(
                cent[lab]
                -
                target_center
            )
        )


        candidates.append(
            (
                d,
                -area,
                lab
            )
        )


    if not candidates:

        return (
            None,
            {
                "quality":
                    "NO_VALID_COMPONENT",

                "bg_med":
                    bg_med,

                "bg_sig":
                    bg_sig,

                "threshold":
                    thr
            }
        )


    lab = min(
        candidates
    )[2]


    comp = (
        labels == lab
    )


    return (
        (
            comp,
            (
                wx1,
                wy1,
                wx2,
                wy2
            )
        ),

        {
            "quality":
                "OK",

            "bg_med":
                bg_med,

            "bg_sig":
                bg_sig,

            "threshold":
                thr
        }
    )


# =============================================================================
# 5. COMPONENT GEOMETRY
# =============================================================================

def component_geometry(
    comp,
    spacing_m
):

    ys, xs = np.where(
        comp
    )


    if len(xs) < 2:

        return {

            "sar_signature_area_m2":
                np.nan,

            "sar_apparent_length_m":
                np.nan,

            "sar_apparent_width_m":
                np.nan,

            "sar_aspect_ratio":
                np.nan,

            "sar_orientation_deg":
                np.nan,

            "sar_compactness":
                np.nan,

            "sar_fill_ratio":
                np.nan,

            "component_pixel_count":
                len(xs)
        }


    pts = np.column_stack(
        [
            xs,
            ys
        ]
    ).astype(
        np.float32
    )


    rect = cv2.minAreaRect(
        pts
    )


    (_, _), (w, h), angle = rect


    length = (
        max(w, h)
        *
        spacing_m
    )


    width = max(
        min(w, h)
        *
        spacing_m,

        spacing_m
    )


    contours = cv2.findContours(
        comp.astype(
            np.uint8
        ),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )[0]


    if contours:

        contour = max(
            contours,
            key=cv2.contourArea
        )

        per = float(
            cv2.arcLength(
                contour,
                True
            )
        )

    else:

        per = np.nan


    area_px = float(
        comp.sum()
    )


    bbox_area = max(
        float(w * h),
        1.0
    )


    if np.isfinite(per):

        compact = float(
            4
            *
            np.pi
            *
            area_px
            /
            (
                per * per
                +
                1e-9
            )
        )

    else:

        compact = np.nan


    return {

        "sar_signature_area_m2":
            area_px
            *
            spacing_m
            *
            spacing_m,

        "sar_apparent_length_m":
            float(length),

        "sar_apparent_width_m":
            float(width),

        "sar_aspect_ratio":
            float(
                length
                /
                max(
                    width,
                    1e-6
                )
            ),

        "sar_orientation_deg":
            float(angle),

        "sar_compactness":
            compact,

        "sar_fill_ratio":
            float(
                area_px
                /
                bbox_area
            ),

        "component_pixel_count":
            int(area_px),
    }


# =============================================================================
# 6. TARGET COMPONENT LAND FRACTION
# =============================================================================

def point_land_fraction_for_component(
    comp,
    window_offsets,
    cache,
    p2ll,
    max_samples=250
):
    """
    Estimate how much of the segmented SAR signature lies on land.

    IMPORTANT OPTIMIZATION:
    p2ll is already built ONCE for the current scene.

    Large components are sampled up to max_samples pixels,
    same concept as previous code.
    """

    ys, xs = np.where(
        comp
    )


    if len(xs) == 0:
        return np.nan


    # -------------------------------------------------------------------------
    # Same subsampling logic as before
    # -------------------------------------------------------------------------

    if len(xs) > max_samples:

        idx = np.linspace(
            0,
            len(xs) - 1,
            max_samples
        ).astype(int)

        xs = xs[idx]
        ys = ys[idx]


    wx1, wy1, _, _ = (
        window_offsets
    )


    # -------------------------------------------------------------------------
    # Component pixels → global SAR coordinates
    # -------------------------------------------------------------------------

    global_x = (
        cache["vv_x_min"]
        +
        wx1
        +
        xs.astype(float)
    )


    global_y = (
        cache["vv_y_min"]
        +
        wy1
        +
        ys.astype(float)
    )


    # -------------------------------------------------------------------------
    # Pixel → lon/lat
    #
    # p2ll itself may be scalar-based, so only this small loop remains.
    # Crucially, the interpolation object is NOT rebuilt.
    # -------------------------------------------------------------------------

    lon = np.empty(
        len(xs),
        dtype=float
    )

    lat = np.empty(
        len(xs),
        dtype=float
    )


    for i in range(
        len(xs)
    ):

        lon[i], lat[i] = p2ll(
            global_x[i],
            global_y[i]
        )


    # -------------------------------------------------------------------------
    # Vectorized lon/lat → UTM transformation
    # -------------------------------------------------------------------------

    x_utm, y_utm = transform_to_utm(
        lon,
        lat
    )


    x_utm = np.asarray(
        x_utm,
        dtype=float
    )

    y_utm = np.asarray(
        y_utm,
        dtype=float
    )


    # -------------------------------------------------------------------------
    # FAST Shapely 2.x path
    #
    # intersects_xy includes polygon interior and boundary.
    # This matches the original within() OR touches() concept.
    # -------------------------------------------------------------------------

    try:

        land_flags = shapely.intersects_xy(
            land_union_utm,
            x_utm,
            y_utm
        )


        return float(
            np.mean(
                land_flags
            )
        )


    except Exception:

        # ---------------------------------------------------------------------
        # Safe fallback for older Shapely
        # ---------------------------------------------------------------------

        on = 0


        for xx, yy in zip(
            x_utm,
            y_utm
        ):

            p = Point(
                float(xx),
                float(yy)
            )


            on += int(
                prepared_land_union_utm
                .covers(
                    p
                )
            )


        return float(
            on
            /
            len(xs)
        )


# =============================================================================
# 7. EXTRACT FEATURES FROM ONE CANDIDATE
#
# IMPORTANT:
# CACHE AND p2ll ARE PASSED INTO THE FUNCTION
# =============================================================================

def extract_candidate_features(
    row,
    cache,
    p2ll
):

    vv = cache[
        "vv_db"
    ]

    vh = cache[
        "vh_db"
    ]


    # =========================================================================
    # A. GLOBAL YOLO BBOX → VV LOCAL BBOX
    # =========================================================================

    bx = [

        row.global_x1
        -
        cache["vv_x_min"],

        row.global_y1
        -
        cache["vv_y_min"],

        row.global_x2
        -
        cache["vv_x_min"],

        row.global_y2
        -
        cache["vv_y_min"]
    ]


    # =========================================================================
    # B. TARGET SEGMENTATION
    # =========================================================================

    seg, info = segment_signature(
        vv,
        bx,
        margin=12
    )


    # =========================================================================
    # C. LOCAL TARGET + BACKGROUND WINDOW
    # =========================================================================

    x1, y1, x2, y2 = [
        int(round(v))
        for v in bx
    ]


    H, W = vv.shape

    m = 12


    wx1 = max(
        0,
        x1 - m
    )

    wy1 = max(
        0,
        y1 - m
    )

    wx2 = min(
        W,
        x2 + m
    )

    wy2 = min(
        H,
        y2 + m
    )


    vv_win = vv[
        wy1:wy2,
        wx1:wx2
    ]


    # =========================================================================
    # D. MATCH SAME GEOGRAPHIC WINDOW IN VH
    # =========================================================================

    ggx1 = (
        cache["vv_x_min"]
        +
        wx1
    )

    ggy1 = (
        cache["vv_y_min"]
        +
        wy1
    )

    ggx2 = (
        cache["vv_x_min"]
        +
        wx2
    )

    ggy2 = (
        cache["vv_y_min"]
        +
        wy2
    )


    vh_win, _ = crop_global(
        vh,
        cache["vh_x_min"],
        cache["vh_y_min"],
        ggx1,
        ggy1,
        ggx2,
        ggy2
    )


    # =========================================================================
    # E. GUARANTEE VV/VH WINDOW ALIGNMENT
    # =========================================================================

    if vh_win.shape != vv_win.shape:

        vh_win = cv2.resize(
            vh_win.astype(np.float32),
            (
                vv_win.shape[1],
                vv_win.shape[0]
            ),
            interpolation=cv2.INTER_NEAREST
        )


    # =========================================================================
    # F. DETECTOR BBOX IN LOCAL WINDOW
    # =========================================================================

    tbx1 = max(
        0,
        x1 - wx1
    )

    tby1 = max(
        0,
        y1 - wy1
    )

    tbx2 = min(
        vv_win.shape[1],
        x2 - wx1
    )

    tby2 = min(
        vv_win.shape[0],
        y2 - wy1
    )


    bgmask = np.ones(
        vv_win.shape,
        dtype=bool
    )


    bgmask[
        tby1:tby2,
        tbx1:tbx2
    ] = False


    # =========================================================================
    # G. TARGET MASK + GEOMETRY
    # =========================================================================

    if seg is not None:

        comp, (
            swx1,
            swy1,
            swx2,
            swy2
        ) = seg


        target_mask = comp


        local_spacing_m = estimate_local_pixel_spacing_m(
            row,
            p2ll
        )

        geom = component_geometry(
            comp,
            local_spacing_m
        )


        # ---------------------------------------------------------------------
        # Optimized target land-overlap
        # ---------------------------------------------------------------------

        target_land = (
            point_land_fraction_for_component(

                comp,

                (
                    swx1,
                    swy1,
                    swx2,
                    swy2
                ),

                cache,

                p2ll
            )
        )


    else:

        target_mask = np.zeros(
            vv_win.shape,
            dtype=bool
        )


        target_mask[
            tby1:tby2,
            tbx1:tbx2
        ] = True


        geom = {

            k: np.nan

            for k in [

                "sar_signature_area_m2",

                "sar_apparent_length_m",

                "sar_apparent_width_m",

                "sar_aspect_ratio",

                "sar_orientation_deg",

                "sar_compactness",

                "sar_fill_ratio",

                "component_pixel_count",
                "sar_pixel_spacing_x_m",
                "sar_pixel_spacing_y_m"
            ]
        }
        geom["geometry_spacing_source"] = "SEGMENTATION_FAILED"
        geom["geometry_method"] = "NOT_AVAILABLE"


        target_land = np.nan


    # =========================================================================
    # H. TARGET / BACKGROUND PIXELS
    # =========================================================================

    vv_target = vv_win[
        target_mask
    ]

    vh_target = vh_win[
        target_mask
    ]


    vv_bg = vv_win[
        bgmask
    ]

    vh_bg = vh_win[
        bgmask
    ]


    # =========================================================================
    # I. ROBUST TARGET/BACKGROUND STATISTICS
    # =========================================================================

    vv_tmed, _, _ = robust_stats(
        vv_target
    )

    vh_tmed, _, _ = robust_stats(
        vh_target
    )


    vv_bmed, _, vv_bsig = (
        robust_stats(
            vv_bg
        )
    )


    vh_bmed, _, _ = robust_stats(
        vh_bg
    )


    # =========================================================================
    # J. GEOMETRY PLAUSIBILITY
    # =========================================================================

    length = geom.get(
        "sar_apparent_length_m",
        np.nan
    )

    width = geom.get(
        "sar_apparent_width_m",
        np.nan
    )

    ar = geom.get(
        "sar_aspect_ratio",
        np.nan
    )


    plausible = bool(

        np.isfinite(length)

        and

        np.isfinite(width)

        and

        np.isfinite(ar)

        and

        CFG[
            "min_apparent_length_m"
        ]
        <=
        length
        <=
        CFG[
            "max_apparent_length_m"
        ]

        and

        CFG[
            "min_apparent_width_m"
        ]
        <=
        width
        <=
        CFG[
            "max_apparent_width_m"
        ]

        and

        CFG[
            "min_sar_aspect_ratio"
        ]
        <=
        ar
        <=
        CFG[
            "max_sar_aspect_ratio"
        ]
    )


    # =========================================================================
    # K. FINAL FEATURE RECORD
    # =========================================================================

    out = {

        "candidate_id":
            row.candidate_id,


        # ---------------------------------------------------------------------
        # Target backscatter
        # ---------------------------------------------------------------------

        "vv_target_median_db":
            vv_tmed,

        "vh_target_median_db":
            vh_tmed,


        "vv_target_mean_db":

            float(
                np.nanmean(
                    vv_target
                )
            )

            if
            np.isfinite(
                vv_target
            ).any()

            else
            np.nan,


        "vh_target_mean_db":

            float(
                np.nanmean(
                    vh_target
                )
            )

            if
            np.isfinite(
                vh_target
            ).any()

            else
            np.nan,


        "vv_target_peak_db":

            float(
                np.nanmax(
                    vv_target
                )
            )

            if
            np.isfinite(
                vv_target
            ).any()

            else
            np.nan,


        "vh_target_peak_db":

            float(
                np.nanmax(
                    vh_target
                )
            )

            if
            np.isfinite(
                vh_target
            ).any()

            else
            np.nan,


        # ---------------------------------------------------------------------
        # Background
        # ---------------------------------------------------------------------

        "vv_background_median_db":
            vv_bmed,

        "vh_background_median_db":
            vh_bmed,


        # ---------------------------------------------------------------------
        # Target/background contrast
        # ---------------------------------------------------------------------

        "vv_target_background_contrast_db":

            (
                vv_tmed
                -
                vv_bmed
            )

            if
            np.isfinite(vv_tmed)
            and
            np.isfinite(vv_bmed)

            else
            np.nan,


        "vh_target_background_contrast_db":

            (
                vh_tmed
                -
                vh_bmed
            )

            if
            np.isfinite(vh_tmed)
            and
            np.isfinite(vh_bmed)

            else
            np.nan,


        "vv_robust_contrast_z":

            (
                vv_tmed
                -
                vv_bmed
            )

            /
            (
                vv_bsig
                +
                1e-6
            )

            if

            np.isfinite(vv_tmed)

            and

            np.isfinite(vv_bmed)

            and

            np.isfinite(vv_bsig)

            else
            np.nan,


        # ---------------------------------------------------------------------
        # Target-land overlap
        # ---------------------------------------------------------------------

        "target_land_overlap_fraction":
            target_land,


        # ---------------------------------------------------------------------
        # Segmentation quality
        # ---------------------------------------------------------------------

        "dimension_quality":
            info.get(
                "quality",
                "UNKNOWN"
            ),


        "geometry_plausible":
            plausible,


        # ---------------------------------------------------------------------
        # Geometry fields
        # ---------------------------------------------------------------------

        **geom,
    }


    return out


print(
    "Optimized STEP 17 functions loaded."
)

print(
    "Scene cache will now be loaded once per scene in STEP 18."
)

In [ ]:
# STEP 17B - V4.1 target-segmentation and apparent-geometry safeguards
#
# These functions override STEP 17 definitions before STEP 18 runs.
# Geometry remains an APPARENT SAR SIGNATURE, not a guaranteed hull dimension.

def segment_signature(vv, bbox_local, margin=12):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox_local]
    H, W = vv.shape
    wx1, wy1 = max(0, x1-margin), max(0, y1-margin)
    wx2, wy2 = min(W, x2+margin), min(H, y2+margin)
    win = vv[wy1:wy2, wx1:wx2]

    if win.size < 20 or np.isfinite(win).sum() < 20:
        return None, {"quality": "FAILED_SMALL_WINDOW"}

    bx1, by1 = max(0, x1-wx1), max(0, y1-wy1)
    bx2, by2 = min(win.shape[1], x2-wx1), min(win.shape[0], y2-wy1)

    mask_bg = np.ones(win.shape, dtype=bool)
    mask_bg[by1:by2, bx1:bx2] = False
    bg_med, bg_mad, bg_sig = robust_stats(win[mask_bg])
    if not np.isfinite(bg_med):
        return None, {"quality": "FAILED_BACKGROUND"}

    thr = bg_med + max(2.5*bg_sig, 2.0)
    binary = (np.isfinite(win) & (win >= thr)).astype(np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

    n, labels, stats, cent = cv2.connectedComponentsWithStats(binary, 8)
    if n <= 1:
        return None, {"quality": "NO_COMPONENT", "bg_med": bg_med, "bg_sig": bg_sig, "threshold": thr}

    detector_mask = np.zeros(win.shape, dtype=bool)
    detector_mask[by1:by2, bx1:bx2] = True
    target_center = np.array([(bx1+bx2)/2.0, (by1+by2)/2.0])
    bbox_diag = max(float(np.hypot(max(1,bx2-bx1), max(1,by2-by1))), 1.0)

    candidates = []
    for lab in range(1, n):
        area = int(stats[lab, cv2.CC_STAT_AREA])
        if area < 2:
            continue
        comp = labels == lab
        overlap_px = int(np.logical_and(comp, detector_mask).sum())
        cx, cy = cent[lab]
        centre_in_expanded_box = (
            (bx1-2 <= cx <= bx2+2) and
            (by1-2 <= cy <= by2+2)
        )

        # V4.1 safeguard: a bright object elsewhere in the local window is not
        # accepted as the ship signature just because it is nearest to the box.
        if overlap_px == 0 and not centre_in_expanded_box:
            continue

        overlap_fraction = overlap_px / max(area, 1)
        distance_norm = float(np.linalg.norm(np.array([cx,cy])-target_center) / bbox_diag)

        # Prefer real intersection with the YOLO box, then a compact centre distance.
        score = (overlap_px, overlap_fraction, -distance_norm, min(area, 500))
        candidates.append((score, lab))

    if not candidates:
        return None, {
            "quality": "NO_COMPONENT_IN_DETECTOR_BOX",
            "bg_med": bg_med,
            "bg_sig": bg_sig,
            "threshold": thr,
        }

    lab = max(candidates, key=lambda z: z[0])[1]
    comp = labels == lab
    yy, xx = np.where(comp)
    touches_border = bool(
        len(xx) and (
            (xx == 0).any() or (yy == 0).any() or
            (xx == win.shape[1]-1).any() or (yy == win.shape[0]-1).any()
        )
    )

    return (
        (comp, (wx1, wy1, wx2, wy2)),
        {
            "quality": "OK_BORDER_TOUCH" if touches_border else "OK",
            "bg_med": bg_med,
            "bg_sig": bg_sig,
            "threshold": thr,
        }
    )


def estimate_local_pixel_spacing_m(row, p2ll):
    """
    Estimate local ground spacing from the Sentinel-1 geolocation interpolator.

    Returns approximate ground distance represented by one image pixel in the
    x and y directions. Falls back to CFG['pixel_spacing_m_approx'] if the
    geolocation estimate is invalid.
    """
    fallback = float(CFG['pixel_spacing_m_approx'])

    try:
        cx = 0.5 * (float(row.global_x1) + float(row.global_x2))
        cy = 0.5 * (float(row.global_y1) + float(row.global_y2))

        lon, lat = p2ll(
            np.array([cx, cx + 1.0, cx], dtype=float),
            np.array([cy, cy, cy + 1.0], dtype=float)
        )
        x, y = transform_to_utm(
            np.asarray(lon, dtype=float),
            np.asarray(lat, dtype=float)
        )
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)

        sx = float(np.hypot(x[1] - x[0], y[1] - y[0]))
        sy = float(np.hypot(x[2] - x[0], y[2] - y[0]))

        # Broad safety limits for Sentinel-1 GRD geolocation interpolation.
        if not (2.0 <= sx <= 40.0):
            sx = fallback
        if not (2.0 <= sy <= 40.0):
            sy = fallback

        return sx, sy

    except Exception:
        return fallback, fallback


def component_geometry(comp, spacing_m):
    """
    Measure the apparent SAR bright signature in approximate ground metres.

    `spacing_m` may be a scalar fallback or `(x_spacing_m, y_spacing_m)`.
    The result remains an apparent SAR-signature geometry, not an exact hull
    measurement.
    """
    ys, xs = np.where(comp)

    if np.isscalar(spacing_m):
        sx = sy = float(spacing_m)
        spacing_source = 'FIXED_FALLBACK'
    else:
        sx, sy = [float(v) for v in spacing_m]
        spacing_source = 'LOCAL_GEOLOCATION'

    if len(xs) < 2:
        return {
            "sar_signature_area_m2": np.nan,
            "sar_apparent_length_m": np.nan,
            "sar_apparent_width_m": np.nan,
            "sar_aspect_ratio": np.nan,
            "sar_orientation_deg": np.nan,
            "sar_compactness": np.nan,
            "sar_fill_ratio": np.nan,
            "component_pixel_count": int(len(xs)),
            "sar_pixel_spacing_x_m": sx,
            "sar_pixel_spacing_y_m": sy,
            "geometry_spacing_source": spacing_source,
            "geometry_method": "MIN_AREA_RECT_GROUND_SCALED_APPROX",
        }

    # Scale pixel centres into local approximate ground coordinates.
    pts_m = np.column_stack([xs * sx, ys * sy]).astype(np.float32)
    (_, _), (w_m, h_m), angle = cv2.minAreaRect(pts_m)

    long_m = max(float(w_m), float(h_m))
    short_m = min(float(w_m), float(h_m))

    # minAreaRect spans pixel centres. Add one representative pixel footprint
    # so a one-pixel-wide target does not collapse toward zero width.
    pixel_pad = 0.5 * (sx + sy)
    length = max(long_m + pixel_pad, pixel_pad)
    width = max(short_m + pixel_pad, min(sx, sy))

    # Normalize orientation to the long axis in [0, 180).
    long_axis_angle = (angle + 90.0) if w_m < h_m else angle
    orientation = float(long_axis_angle % 180.0)

    contours = cv2.findContours(
        comp.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )[0]

    perimeter_m = np.nan
    if contours:
        contour = max(contours, key=cv2.contourArea).reshape(-1, 2).astype(np.float32)
        contour_m = contour.copy()
        contour_m[:, 0] *= sx
        contour_m[:, 1] *= sy
        perimeter_m = float(cv2.arcLength(contour_m.reshape(-1, 1, 2), True))

    area_m2 = float(comp.sum()) * sx * sy
    # Use the padded apparent rectangle for occupancy. This keeps fill_ratio
    # physically interpretable and bounded by 1 even though minAreaRect spans
    # pixel centres while area uses full pixel footprints.
    rect_area_m2 = max(float(length * width), area_m2, sx * sy)

    compactness = (
        float(4.0 * np.pi * area_m2 / (perimeter_m * perimeter_m + 1e-9))
        if np.isfinite(perimeter_m) and perimeter_m > 0
        else np.nan
    )

    return {
        "sar_signature_area_m2": area_m2,
        "sar_apparent_length_m": float(length),
        "sar_apparent_width_m": float(width),
        "sar_aspect_ratio": float(length / max(width, 1e-6)),
        "sar_orientation_deg": orientation,
        "sar_compactness": compactness,
        "sar_fill_ratio": float(area_m2 / max(rect_area_m2, 1e-6)),
        "component_pixel_count": int(comp.sum()),
        "sar_pixel_spacing_x_m": sx,
        "sar_pixel_spacing_y_m": sy,
        "geometry_spacing_source": spacing_source,
        "geometry_method": "MIN_AREA_RECT_GROUND_SCALED_APPROX",
    }


def point_land_fraction_for_component(
    comp,
    window_offsets,
    cache,
    p2ll,
    max_samples=160
):
    """
    V4.1 vectorized target-land overlap.

    Up to 160 component pixels are sampled deterministically. Pixel -> lon/lat
    interpolation is vectorized using the once-per-scene interpolator, then the
    Shapely land test is vectorized. This keeps the quantity diagnostic while
    avoiding hundreds of scalar interpolation calls per target.
    """
    ys, xs = np.where(comp)
    if len(xs) == 0:
        return np.nan

    if len(xs) > max_samples:
        idx = np.linspace(0, len(xs)-1, max_samples).astype(int)
        xs = xs[idx]
        ys = ys[idx]

    wx1, wy1, _, _ = window_offsets
    global_x = cache["vv_x_min"] + wx1 + xs.astype(float)
    global_y = cache["vv_y_min"] + wy1 + ys.astype(float)

    lon, lat = p2ll(global_x, global_y)
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)

    x_utm, y_utm = transform_to_utm(lon, lat)
    x_utm = np.asarray(x_utm, dtype=float)
    y_utm = np.asarray(y_utm, dtype=float)

    try:
        land_flags = shapely.intersects_xy(
            land_union_utm,
            x_utm,
            y_utm
        )
        return float(np.mean(land_flags))
    except Exception:
        flags = [
            bool(land_union_utm.covers(Point(float(xx), float(yy))))
            for xx, yy in zip(x_utm, y_utm)
        ]
        return float(np.mean(flags)) if flags else np.nan

print("V4.1 vectorized target-land overlap active.")

print("V4.1 target segmentation safeguard active.")
print("Target dimensions remain apparent SAR-signature dimensions, not guaranteed hull dimensions.")


In [ ]:
# =============================================================================
# STEP 18 - FAST SCENE-WISE SAR FEATURE EXTRACTION
#
# Key changes:
#   1. Load each Sentinel-1 scene ONCE
#   2. Build pixel→lon/lat converter ONCE per scene
#   3. Reuse VV/VH arrays for every candidate in that scene
#   4. tqdm gives candidate + scene ETA
#   5. Checkpoint after each scene
#   6. Additional checkpoint every N candidates inside a large scene
#   7. Resume completed candidates automatically
# =============================================================================

import gc
import time
import numpy as np
import pandas as pd

from tqdm.auto import tqdm


print("=" * 100)
print("STEP 18 - FAST SAR TARGET FEATURE EXTRACTION")
print("=" * 100)


STEP_START = time.perf_counter()


# =============================================================================
# CONFIG
# =============================================================================

# Save partial progress within large scenes.
# 200 is a good balance between safety and excessive disk writes.
CHECKPOINT_EVERY = 200


# =============================================================================
# 1. LOAD EXISTING CHECKPOINT
# =============================================================================

if FORCE["sar_features"]:

    print(
        "FORCE sar_features = True"
    )

    print(
        "Existing SAR feature results will be recalculated."
    )

    sar_features = pd.DataFrame()


else:

    sar_features = load_table(
        "05_sar_features/SAR_CANDIDATE_FEATURES_V4_1"
    )


    if sar_features is None:

        sar_features = pd.DataFrame()


# =============================================================================
# 2. NORMALIZE EXISTING IDs
# =============================================================================

if len(sar_features):

    sar_features[
        "candidate_id"
    ] = (

        sar_features[
            "candidate_id"
        ]

        .astype(str)
    )


    completed_ids = set(

        sar_features[
            "candidate_id"
        ]
    )


else:

    completed_ids = set()


print(
    "Already completed candidates :",
    len(completed_ids)
)

print(
    "Total fused candidates       :",
    len(fused)
)


# =============================================================================
# 3. DETERMINE MISSING CANDIDATES
# =============================================================================

work = fused.copy()


work[
    "_candidate_id_str"
] = (

    work[
        "candidate_id"
    ]

    .astype(str)
)


if FORCE["sar_features"]:

    need = work.copy()


else:

    need = work[

        ~work[
            "_candidate_id_str"
        ]

        .isin(
            completed_ids
        )

    ].copy()


print(
    "Candidates still to process  :",
    len(need)
)


# =============================================================================
# 4. HELPER FOR SAVING PARTIAL RESULTS
# =============================================================================

def append_and_checkpoint(
    current,
    new_rows
):

    if not new_rows:

        return current


    new_df = pd.DataFrame(
        new_rows
    )


    new_df[
        "candidate_id"
    ] = (

        new_df[
            "candidate_id"
        ]

        .astype(str)
    )


    if len(current):

        current = pd.concat(

            [
                current,
                new_df
            ],

            ignore_index=True
        )


    else:

        current = (
            new_df.copy()
        )


    current[
        "candidate_id"
    ] = (

        current[
            "candidate_id"
        ]

        .astype(str)
    )


    current = (

        current

        .drop_duplicates(
            subset="candidate_id",
            keep="last"
        )

        .reset_index(
            drop=True
        )
    )


    save_table(

        current,

        DIRS[
            "sar_features"
        ]

        /
        "SAR_CANDIDATE_FEATURES_V4_1"
    )


    return current


# =============================================================================
# 5. NOTHING LEFT?
# =============================================================================

if len(need) == 0:

    print()

    print(
        "All fused candidates already have SAR features."
    )

    print(
        "No SAR feature extraction required."
    )


# =============================================================================
# 6. PROCESS SCENE BY SCENE
# =============================================================================

else:

    scene_groups = list(

        need.groupby(
            "scene_id",
            sort=False
        )
    )


    print(
        "Scenes requiring SAR extraction:",
        len(scene_groups)
    )

    print()


    scene_bar = tqdm(

        scene_groups,

        total=
            len(scene_groups),

        desc=
            "SAR feature scenes",

        unit=
            "scene",

        dynamic_ncols=
            True
    )


    for (
        scene_number,
        (
            scene_id,
            scene_df
        )
    ) in enumerate(

        scene_bar,
        start=1
    ):


        scene_start = (
            time.perf_counter()
        )


        scene_bar.set_postfix_str(

            f"{scene_id} | "
            f"{len(scene_df)} candidates"
        )


        print(
            "\n"
            +
            "-" * 100
        )


        print(
            f"SCENE "
            f"{scene_number}"
            f"/"
            f"{len(scene_groups)}"
        )


        print(
            "Scene ID   :",
            scene_id
        )


        print(
            "Candidates :",
            len(scene_df)
        )


        # =====================================================================
        # A. LOAD SCENE CACHE ONCE
        # =====================================================================

        load_start = (
            time.perf_counter()
        )


        try:

            cache = load_scene_cache(
                scene_id
            )


            p2ll = (
                build_pixel_to_lonlat(
                    cache[
                        "geo_points"
                    ]
                )
            )


        except Exception as e:

            print(
                "ERROR loading scene:",
                scene_id,
                repr(e)
            )

            continue


        load_elapsed = (
            time.perf_counter()
            -
            load_start
        )


        print(
            f"Scene cache loaded in "
            f"{load_elapsed:.2f} s"
        )


        print(
            "VV shape:",
            cache["vv_db"].shape
        )


        print(
            "VH shape:",
            cache["vh_db"].shape
        )


        # =====================================================================
        # B. PROCESS ALL CANDIDATES USING SAME CACHE
        # =====================================================================

        pending_rows = []

        scene_completed = 0

        errors = 0


        candidate_bar = tqdm(

            scene_df.itertuples(
                index=False
            ),

            total=
                len(scene_df),

            desc=
                f"  {scene_id}",

            unit=
                "candidate",

            leave=
                False,

            dynamic_ncols=
                True
        )


        for i, row in enumerate(
            candidate_bar,
            start=1
        ):


            try:

                result = (
                    extract_candidate_features(

                        row,

                        cache,

                        p2ll
                    )
                )


                if result is not None:

                    pending_rows.append(
                        result
                    )

                    scene_completed += 1


                else:

                    errors += 1


            except Exception as e:

                errors += 1


                if errors <= 5:

                    print(
                        "\nSAR feature error:",
                        row.candidate_id,
                        repr(e)
                    )


            # =================================================================
            # C. MID-SCENE CHECKPOINT
            # =================================================================

            if (
                len(pending_rows)
                >=
                CHECKPOINT_EVERY
            ):

                checkpoint_start = (
                    time.perf_counter()
                )


                sar_features = (
                    append_and_checkpoint(

                        sar_features,

                        pending_rows
                    )
                )


                checkpoint_elapsed = (

                    time.perf_counter()
                    -
                    checkpoint_start
                )


                candidate_bar.set_postfix_str(

                    f"saved {len(sar_features)} | "
                    f"checkpoint {checkpoint_elapsed:.1f}s"
                )


                pending_rows = []


        # =====================================================================
        # D. SAVE REMAINDER FROM CURRENT SCENE
        # =====================================================================

        if pending_rows:

            sar_features = (
                append_and_checkpoint(

                    sar_features,

                    pending_rows
                )
            )


            pending_rows = []


        # =====================================================================
        # E. SCENE TIMING SUMMARY
        # =====================================================================

        scene_elapsed = (

            time.perf_counter()
            -
            scene_start
        )


        processing_elapsed = max(

            scene_elapsed
            -
            load_elapsed,

            1e-9
        )


        speed = (

            scene_completed
            /
            processing_elapsed
        )


        sec_per_candidate = (

            processing_elapsed
            /
            max(
                scene_completed,
                1
            )
        )


        print()

        print(
            f"Completed              : "
            f"{scene_completed}"
            f"/"
            f"{len(scene_df)}"
        )


        print(
            f"Errors                 : "
            f"{errors}"
        )


        print(
            f"Scene load time        : "
            f"{load_elapsed:.2f} sec"
        )


        print(
            f"Total scene time       : "
            f"{scene_elapsed:.2f} sec"
        )


        print(
            f"Candidates/sec         : "
            f"{speed:.2f}"
        )


        print(
            f"Seconds/candidate      : "
            f"{sec_per_candidate:.3f}"
        )


        print(
            "Checkpoint             :",
            DIRS[
                "sar_features"
            ]
            /
            "SAR_CANDIDATE_FEATURES_V4_1"
        )


        # =====================================================================
        # F. MEMORY CLEANUP
        # =====================================================================

        del cache
        del p2ll


        gc.collect()


# =============================================================================
# 7. RELOAD FINAL SAVED TABLE
# =============================================================================

sar_final = load_table(
    "05_sar_features/SAR_CANDIDATE_FEATURES_V4_1"
)


if sar_final is not None:

    sar_features = (
        sar_final.copy()
    )


# =============================================================================
# 8. RETAIN ONLY CURRENT FUSED CANDIDATES
# =============================================================================

if len(sar_features):

    sar_features[
        "candidate_id"
    ] = (

        sar_features[
            "candidate_id"
        ]

        .astype(str)
    )


    valid_ids = set(

        fused[
            "candidate_id"
        ]

        .astype(str)
    )


    sar_features = (

        sar_features[

            sar_features[
                "candidate_id"
            ]

            .isin(
                valid_ids
            )
        ]

        .copy()
    )


# =============================================================================
# 9. FINAL QA
# =============================================================================

total_elapsed = (

    time.perf_counter()
    -
    STEP_START
)


print(
    "\n"
    +
    "=" * 100
)


print(
    "SAR TARGET FEATURE EXTRACTION COMPLETE"
)


print(
    "=" * 100
)


print(
    "Input fused candidates :",
    len(fused)
)


print(
    "SAR feature rows       :",
    len(sar_features)
)


print(
    f"Total elapsed time     : "
    f"{total_elapsed / 60:.2f} minutes"
)


if len(fused):

    completeness = (

        100
        *
        len(sar_features)
        /
        len(fused)
    )


    print(
        f"Feature completeness   : "
        f"{completeness:.2f}%"
    )


# =============================================================================
# 10. FEATURE QUALITY SUMMARY
# =============================================================================

if len(sar_features):


    if (
        "dimension_quality"
        in
        sar_features.columns
    ):

        print(
            "\nSegmentation / dimension quality:"
        )


        print(

            sar_features[
                "dimension_quality"
            ]

            .value_counts(
                dropna=False
            )
        )


    if (
        "geometry_plausible"
        in
        sar_features.columns
    ):

        print(
            "\nGeometry plausible:"
        )


        print(

            sar_features[
                "geometry_plausible"
            ]

            .value_counts(
                dropna=False
            )
        )


    if (
        "target_land_overlap_fraction"
        in
        sar_features.columns
    ):

        valid_land = (

            sar_features[
                "target_land_overlap_fraction"
            ]

            .notna()
        )


        print(
            "\nCandidates with target-land estimate:",
            int(
                valid_land.sum()
            )
        )


        print(
            "Candidates with target signature touching land:",
            int(

                (
                    sar_features[
                        "target_land_overlap_fraction"
                    ]
                    >
                    0
                )

                .sum()
            )
        )


# =============================================================================
# 11. OUTPUT LOCATION
# =============================================================================

print(
    "\nCheckpoint:"
)


print(

    DIRS[
        "sar_features"
    ]

    /
    "SAR_CANDIDATE_FEATURES_V4_1"
)


stage_complete(

    "SAR TARGET PROPERTIES",

    input_rows=
        len(fused),

    output_rows=
        len(sar_features),

    checkpoint=
        DIRS[
            "sar_features"
        ]
)

# Part H - AIS temporal handling and one-to-one association

Global Fishing Watch `public-global-presence:latest` provides one AIS-presence position per hour per vessel. V4 uses two modes:

- `INTERPOLATED_HOURLY_POINTS`: a vessel has hourly points before and after the SAR time and an approximate linear interpolation is possible.
- `NEAREST_HOURLY_PRESENCE`: only the nearest hourly observation is available within the configured tolerance.

This is more defensible than matching every SAR target independently to any AIS cell from a broad multi-hour/day window.

In [ ]:
# STEP 19 - GFW token, API cache, temporal interpolation and one-to-one matching
_GFW_TOKEN=None

def get_gfw_token():
    global _GFW_TOKEN
    if _GFW_TOKEN: return _GFW_TOKEN
    # Prefer Kaggle secret if available, otherwise prompt securely.
    try:
        from kaggle_secrets import UserSecretsClient
        tok=UserSecretsClient().get_secret('fish')
    except Exception:
        tok=None
    if not tok:
        tok=getpass('Global Fishing Watch API token: ')
    if not tok:
        raise RuntimeError('GFW token is required for uncached AIS retrieval.')
    _GFW_TOKEN=tok.strip()
    return _GFW_TOKEN


def flatten_gfw_response(payload):
    records=[]
    for entry in payload.get('entries',[]):
        if not isinstance(entry,dict): continue
        scalar={k:v for k,v in entry.items() if not isinstance(v,(list,dict))}
        lists=[(k,v) for k,v in entry.items() if isinstance(v,list)]
        if lists:
            for key,vals in lists:
                for row in vals:
                    if isinstance(row,dict):
                        rec={**scalar,**row,'source_key':key}; records.append(rec)
        else:
            records.append(scalar)
    return pd.DataFrame(records)


def fetch_gfw_scene(scene_row):
    raw_json=DIRS['ais']/'raw_reports'/f'{scene_row.scene_id}_gfw.json'
    parsed_csv=DIRS['ais']/'per_scene'/f'{scene_row.scene_id}_raw_ais.csv'
    old_json=read_path(Path('06_ais/raw_reports')/raw_json.name)
    old_csv=read_path(Path('06_ais/per_scene')/parsed_csv.name)
    if old_csv.exists() and old_json.exists() and not FORCE['ais']:
        return pd.read_csv(old_csv)

    acq=pd.to_datetime(scene_row.start_time,utc=True)
    start=(acq-pd.Timedelta(hours=CFG['ais_query_window_hours'])).strftime('%Y-%m-%d')
    end=(acq+pd.Timedelta(hours=CFG['ais_query_window_hours'])+pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    params=[
        ('spatial-resolution','HIGH'),('temporal-resolution','HOURLY'),('spatial-aggregation','false'),
        ('datasets[0]',CFG['gfw_dataset']),('date-range',f'{start},{end}'),('format','JSON'),('group-by','VESSEL_ID')
    ]
    headers={'Authorization':f'Bearer {get_gfw_token()}','Content-Type':'application/json'}
    resp=requests.post(CFG['gfw_report_url'],headers=headers,params=params,json={'geojson':AOI_GEOJSON},timeout=180)
    if resp.status_code!=200:
        raise RuntimeError(f'GFW AIS request failed {resp.status_code}: {resp.text[:500]}')
    payload=resp.json(); save_json(raw_json,payload)
    df=flatten_gfw_response(payload)
    df.to_csv(parsed_csv,index=False)
    return df


def infer_id_column(df):
    for c in ['vesselId','vessel_id','mmsi','id']:
        if c in df.columns: return c
    return None


def ais_positions_at_sar(raw_df, acquisition):
    if raw_df is None or len(raw_df)==0:
        return pd.DataFrame()
    df=raw_df.copy()
    if 'date' not in df.columns or 'lat' not in df.columns or 'lon' not in df.columns:
        return pd.DataFrame()
    df['bin_time']=pd.to_datetime(df['date'],utc=True,errors='coerce')
    df['lat']=pd.to_numeric(df['lat'],errors='coerce'); df['lon']=pd.to_numeric(df['lon'],errors='coerce')
    df=df.dropna(subset=['bin_time','lat','lon'])
    idc=infer_id_column(df)
    if idc is None:
        df['_tmp_id']=np.arange(len(df)).astype(str); idc='_tmp_id'
    acq=pd.to_datetime(acquisition,utc=True)
    rows=[]
    for vid,g in df.groupby(idc):
        g=g.sort_values('bin_time')
        before=g[g.bin_time<=acq]
        after=g[g.bin_time>=acq]
        b=before.iloc[-1] if len(before) else None
        a=after.iloc[0] if len(after) else None
        rec=None
        if b is not None and a is not None and b.bin_time!=a.bin_time:
            db=abs((acq-b.bin_time).total_seconds())/60
            da=abs((a.bin_time-acq).total_seconds())/60
            if db<=90 and da<=90:
                total=(a.bin_time-b.bin_time).total_seconds()
                alpha=(acq-b.bin_time).total_seconds()/total if total>0 else 0
                rec=dict(b)
                rec['lat_at_sar']=float(b.lat + alpha*(a.lat-b.lat))
                rec['lon_at_sar']=float(b.lon + alpha*(a.lon-b.lon))
                rec['ais_time_mode']='INTERPOLATED_HOURLY_POINTS'
                rec['sar_ais_time_difference_min']=float(max(db,da))
                rec['ais_before_time']=pd.to_datetime(b.bin_time,utc=True).isoformat()
                rec['ais_after_time']=pd.to_datetime(a.bin_time,utc=True).isoformat()
                rec['ais_source_time']=None
        if rec is None:
            gg=g.copy(); gg['dtmin']=(gg.bin_time-acq).abs().dt.total_seconds()/60
            q=gg.sort_values('dtmin').iloc[0]
            if q.dtmin<=CFG['ais_nearest_point_max_minutes']:
                rec=dict(q); rec['lat_at_sar']=float(q.lat); rec['lon_at_sar']=float(q.lon)
                rec['ais_time_mode']='NEAREST_HOURLY_PRESENCE'; rec['sar_ais_time_difference_min']=float(q.dtmin)
                rec['ais_before_time']=None; rec['ais_after_time']=None
                rec['ais_source_time']=pd.to_datetime(q.bin_time,utc=True).isoformat()
        if rec is not None:
            rec['matched_ais_id']=str(vid)
            rows.append(rec)
    return pd.DataFrame(rows)


def haversine_matrix(lat1,lon1,lat2,lon2):
    R=6371000.0
    lat1=np.radians(np.asarray(lat1))[:,None]; lon1=np.radians(np.asarray(lon1))[:,None]
    lat2=np.radians(np.asarray(lat2))[None,:]; lon2=np.radians(np.asarray(lon2))[None,:]
    dlat=lat2-lat1; dlon=lon2-lon1
    a=np.sin(dlat/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arctan2(np.sqrt(a),np.sqrt(1-a))


def one_to_one_ais(scene_candidates, aispos):
    base=pd.DataFrame({'candidate_id':scene_candidates.candidate_id})
    for c,default in [
        ('ais_matched',False),('nearest_ais_distance_m',np.nan),('matched_ais_id',None),('matched_vessel_name',None),
        ('matched_vessel_length_m',np.nan),('sar_ais_time_difference_min',np.nan),('ais_time_mode',None),('ais_match_quality','NONE'),
        ('ais_before_time',None),('ais_after_time',None),('ais_source_time',None)]:
        base[c]=default
    if len(scene_candidates)==0 or aispos is None or len(aispos)==0:
        return base
    D=haversine_matrix(scene_candidates.latitude.values,scene_candidates.longitude.values,aispos.lat_at_sar.values,aispos.lon_at_sar.values)
    rr,cc=linear_sum_assignment(D)
    for i,j in zip(rr,cc):
        d=float(D[i,j])
        if d>CFG['ais_hard_match_max_m']: continue
        ap=aispos.iloc[j]
        mode=ap.get('ais_time_mode',None); dt=float(ap.get('sar_ais_time_difference_min',np.nan))
        if mode=='INTERPOLATED_HOURLY_POINTS' and d<=CFG['ais_interp_high_m']:
            qual='HIGH'
        elif mode=='INTERPOLATED_HOURLY_POINTS' and d<=CFG['ais_interp_moderate_m']:
            qual='MODERATE'
        elif mode=='NEAREST_HOURLY_PRESENCE' and d<=CFG['ais_nearest_high_m'] and dt<=30:
            qual='HIGH'
        elif mode=='NEAREST_HOURLY_PRESENCE' and d<=CFG['ais_nearest_moderate_m'] and dt<=45:
            qual='MODERATE'
        else:
            qual='LOW'
        idx=base.index[i]
        base.loc[idx,'ais_matched']=True
        base.loc[idx,'nearest_ais_distance_m']=d
        base.loc[idx,'matched_ais_id']=str(ap.get('matched_ais_id',''))
        name=ap.get('shipName',ap.get('ship_name',None)); base.loc[idx,'matched_vessel_name']=name
        length=ap.get('length',ap.get('vesselLength',np.nan)); base.loc[idx,'matched_vessel_length_m']=pd.to_numeric(length,errors='coerce')
        base.loc[idx,'sar_ais_time_difference_min']=dt
        base.loc[idx,'ais_time_mode']=mode
        base.loc[idx,'ais_match_quality']=qual
        base.loc[idx,'ais_before_time']=ap.get('ais_before_time',None)
        base.loc[idx,'ais_after_time']=ap.get('ais_after_time',None)
        base.loc[idx,'ais_source_time']=ap.get('ais_source_time',None)
    return base

In [ ]:
# ============================================================
# STEP 20 - AIS retrieval and association per scene
# CORRECTED VERSION
# ============================================================

# Clean V4.1 work root: reuse valid local/compatible V4.1 AIS checkpoints unless FORCE['ais']=True.
REBUILD_AIS_ASSOCIATION = bool(FORCE['ais'])

AIS_STEM = Path('06_ais/AIS_ASSOCIATION')

# ------------------------------------------------------------
# 1. Load previous AIS association only if we are NOT rebuilding
# ------------------------------------------------------------
if REBUILD_AIS_ASSOCIATION:
    ais_old = pd.DataFrame()
else:
    ais_old = None if FORCE['ais'] else load_table(AIS_STEM)

    if ais_old is None:
        ais_old = pd.DataFrame()


# ------------------------------------------------------------
# 2. Determine scenes that need AIS processing
# ------------------------------------------------------------
old_scene_ids = (
    set(ais_old.scene_id.unique())
    if len(ais_old) and 'scene_id' in ais_old.columns
    else set()
)

current_scene_ids = (
    set(fused.scene_id.unique())
    if len(fused)
    else set()
)

if REBUILD_AIS_ASSOCIATION or FORCE['ais']:
    target_scene_ids = sorted(current_scene_ids)
else:
    target_scene_ids = sorted(
        current_scene_ids - old_scene_ids
    )


# ------------------------------------------------------------
# 3. Initialise output containers
# ------------------------------------------------------------
if REBUILD_AIS_ASSOCIATION or FORCE['ais']:
    rows = []
else:
    rows = [ais_old] if len(ais_old) else []

errors = []


print("AIS scenes to process:", len(target_scene_ids))


# ------------------------------------------------------------
# 4. Process AIS scene-by-scene
# ------------------------------------------------------------
if target_scene_ids:

    catalog_by_id = scene_catalog.set_index('scene_id')

    for sid in target_scene_ids:

        print("\nProcessing AIS:", sid)

        # ----------------------------------------------------
        # IMPORTANT FIX
        # scene_id disappears as a normal column after
        # set_index('scene_id'), so restore it explicitly.
        # ----------------------------------------------------
        scene_row = catalog_by_id.loc[sid].copy()
        scene_row['scene_id'] = sid

        scene_fused = fused[
            fused.scene_id == sid
        ].copy()

        try:

            # -----------------------------------------------
            # Retrieve Global Fishing Watch AIS information
            # Existing cached AIS files can still be reused.
            # -----------------------------------------------
            raw = fetch_gfw_scene(scene_row)

            # -----------------------------------------------
            # Estimate AIS vessel positions at SAR acquisition
            # -----------------------------------------------
            aispos = ais_positions_at_sar(
                raw,
                scene_row.start_time
            )

            # -----------------------------------------------
            # One-to-one SAR candidate ↔ AIS association
            # -----------------------------------------------
            assoc = one_to_one_ais(
                scene_fused,
                aispos
            )

            assoc['scene_id'] = sid

            rows.append(assoc)

            # Save AIS positions used for this scene
            aispos.to_csv(
                DIRS['ais'] / 'per_scene' /
                f'{sid}_positions_at_sar.csv',
                index=False
            )

            print(
                sid,
                'AIS positions:',
                len(aispos),
                'matched:',
                int(assoc.ais_matched.sum())
            )

        except Exception as e:

            # -----------------------------------------------
            # Record actual AIS failure
            # -----------------------------------------------
            errors.append({
                'scene_id': sid,
                'error': repr(e)
            })

            print(
                'AIS ERROR',
                sid,
                repr(e)
            )

            # Preserve candidates even if AIS retrieval failed
            empty = one_to_one_ais(
                scene_fused,
                pd.DataFrame()
            )

            empty['scene_id'] = sid

            rows.append(empty)


# ------------------------------------------------------------
# 5. Combine all scene-level AIS associations
# ------------------------------------------------------------
ais_assoc = (
    pd.concat(rows, ignore_index=True)
    if rows
    else pd.DataFrame()
)


# ------------------------------------------------------------
# 6. Ensure one AIS association record per candidate
# ------------------------------------------------------------
if len(ais_assoc):

    ais_assoc = ais_assoc.drop_duplicates(
        'candidate_id',
        keep='last'
    )


# ------------------------------------------------------------
# 7. Save AIS association checkpoint
# ------------------------------------------------------------
save_table(
    ais_assoc,
    DIRS['ais'] / 'AIS_ASSOCIATION'
)


# ------------------------------------------------------------
# 8. Save real AIS errors
# ------------------------------------------------------------
ais_error_file = DIRS['logs'] / 'ais_errors.csv'

if errors:

    pd.DataFrame(errors).to_csv(
        ais_error_file,
        index=False
    )

else:

    # Remove stale error file from previous failed run
    if ais_error_file.exists():
        ais_error_file.unlink()


# ------------------------------------------------------------
# 9. Keep only candidates still present in fused dataset
# ------------------------------------------------------------
if len(ais_assoc):

    ais_assoc = ais_assoc[
        ais_assoc.candidate_id.isin(
            fused.candidate_id
        )
    ].copy()


# ------------------------------------------------------------
# 10. Stage summary
# ------------------------------------------------------------
stage_complete(
    'AIS ASSOCIATION',
    input_rows=len(fused),
    output_rows=len(ais_assoc),
    checkpoint=DIRS['ais'],
    errors=len(errors)
)


# ------------------------------------------------------------
# 11. Quick AIS QA summary
# ------------------------------------------------------------
if len(ais_assoc):

    n_matched = int(
        ais_assoc['ais_matched'].fillna(False).sum()
    )

    print("\nAIS ASSOCIATION SUMMARY")
    print("-----------------------")
    print("Candidates :", len(ais_assoc))
    print("AIS matched:", n_matched)
    print(
        "Unmatched  :",
        len(ais_assoc) - n_matched
    )

print("AIS errors :", len(errors))

In [ ]:
# STEP 20B - AIS-centred detector coverage audit
#
# This is NOT formal recall until expected_sar_visible is manually reviewed.
# It tells you whether an AIS-supported vessel had any fused SAR candidate
# nearby, which is the first diagnostic for "coastal ship was not detected".

def nearest_candidate_to_ais(scene_candidates, aispos):
    if aispos is None or len(aispos)==0:
        return pd.DataFrame()
    out=aispos.copy().reset_index(drop=True)

    if scene_candidates is None or len(scene_candidates)==0:
        out['nearest_candidate_id']=None
        out['nearest_candidate_distance_m']=np.nan
        out['candidate_within_300m']=False
        out['candidate_within_500m']=False
        return out

    D=haversine_matrix(
        out.lat_at_sar.values,
        out.lon_at_sar.values,
        scene_candidates.latitude.values,
        scene_candidates.longitude.values
    )
    j=D.argmin(axis=1)
    d=D[np.arange(len(out)),j]

    out['nearest_candidate_id']=scene_candidates.iloc[j].candidate_id.values
    out['nearest_candidate_distance_m']=d
    out['candidate_within_300m']=d<=300
    out['candidate_within_500m']=d<=500
    return out

audit_rows=[]
for row in allowed_catalog.itertuples(index=False):
    pos_path=DIRS['ais']/'per_scene'/f'{row.scene_id}_positions_at_sar.csv'
    if not pos_path.exists():
        old=read_path(Path('06_ais/per_scene')/pos_path.name)
        pos_path=old if old.exists() else pos_path
    if not pos_path.exists():
        continue

    aispos=pd.read_csv(pos_path)
    if len(aispos)==0:
        continue

    scene_fused=fused[fused.scene_id==row.scene_id].copy()
    a=nearest_candidate_to_ais(scene_fused,aispos)
    a['scene_id']=row.scene_id
    a['scene_datetime']=pd.to_datetime(row.start_time,utc=True).isoformat()
    a['experiment_set']=row.experiment_set
    audit_rows.append(a)

ais_detector_audit=pd.concat(audit_rows,ignore_index=True) if audit_rows else pd.DataFrame()

if len(ais_detector_audit):
    # Manual fields required before this is interpreted as true recall.
    ais_detector_audit['expected_sar_visible']=''
    ais_detector_audit['manual_review_label']=''
    ais_detector_audit['review_notes']=''

    keep=[c for c in [
        'scene_id','scene_datetime','experiment_set','matched_ais_id',
        'shipName','ship_name','lat_at_sar','lon_at_sar','ais_time_mode',
        'sar_ais_time_difference_min','nearest_candidate_id',
        'nearest_candidate_distance_m','candidate_within_300m',
        'candidate_within_500m','expected_sar_visible',
        'manual_review_label','review_notes'
    ] if c in ais_detector_audit.columns]

    ais_detector_audit[keep].to_csv(
        DIRS['validation']/'AIS_CENTERED_DETECTOR_AUDIT.csv',
        index=False
    )

    print("AIS-centred detector audit rows:",len(ais_detector_audit))
    print("AIS positions with candidate <=300 m:",
          f"{100*ais_detector_audit.candidate_within_300m.mean():.1f}%")
    print("AIS positions with candidate <=500 m:",
          f"{100*ais_detector_audit.candidate_within_500m.mean():.1f}%")
    print("Do not call these recall values until expected_sar_visible is reviewed.")


# Part I - Development-only multi-temporal persistence

Recurrence alone is not treated as a fixed object because anchorages can repeatedly contain different vessels. V4 therefore adds positional dispersion and requires recurrence across multiple dates and years before a location can provide strong static-object evidence.

In [ ]:

# STEP 21 - V4.1 persistence model fitting and attachment
TO_UTM=Transformer.from_crs(4326,CFG['utm_epsg'],always_xy=True)

def add_xy(df):
    out=df.copy()
    x,y=TO_UTM.transform(out.longitude.values,out.latitude.values)
    out['utm_x']=x
    out['utm_y']=y
    return out

def axial_orientation_variability_deg(values):
    a=pd.to_numeric(values,errors='coerce').dropna().values.astype(float)
    if len(a) < 2:
        return np.nan
    # Orientation is axial: 0° and 180° are equivalent.
    theta=np.deg2rad(2.0*a)
    C=np.mean(np.cos(theta)); S=np.mean(np.sin(theta))
    R=np.hypot(C,S)
    if R <= 1e-9:
        return 90.0
    return float(np.rad2deg(np.sqrt(max(0.0,-2.0*np.log(min(R,1.0)))))/2.0)

def robust_relative_iqr(values):
    x=pd.to_numeric(values,errors='coerce').dropna().values.astype(float)
    x=x[np.isfinite(x) & (x>0)]
    if len(x) < 3:
        return np.nan
    med=np.median(x)
    return float((np.percentile(x,75)-np.percentile(x,25))/max(med,1e-6))

def fit_persistence_clusters(dev_df):
    d=add_xy(dev_df)
    if len(d)==0:
        return pd.DataFrame()

    labels=DBSCAN(
        eps=CFG['persistence_eps_m'],
        min_samples=CFG['persistence_min_samples']
    ).fit_predict(d[['utm_x','utm_y']].values)
    d['cluster_label']=labels

    rows=[]
    for lab,g in d[d.cluster_label>=0].groupby('cluster_label'):
        cx=float(np.median(g.utm_x)); cy=float(np.median(g.utm_y))
        radial=np.sqrt((g.utm_x-cx)**2+(g.utm_y-cy)**2)

        orient_var=axial_orientation_variability_deg(
            g['sar_orientation_deg'] if 'sar_orientation_deg' in g else pd.Series(dtype=float)
        )
        length_iqr_rel=robust_relative_iqr(
            g['sar_apparent_length_m'] if 'sar_apparent_length_m' in g else pd.Series(dtype=float)
        )
        width_iqr_rel=robust_relative_iqr(
            g['sar_apparent_width_m'] if 'sar_apparent_width_m' in g else pd.Series(dtype=float)
        )

        geometry_consistent = bool(
            (not np.isfinite(orient_var) or orient_var <= 25.0)
            and (not np.isfinite(length_iqr_rel) or length_iqr_rel <= 0.60)
            and (not np.isfinite(width_iqr_rel) or width_iqr_rel <= 0.75)
        )

        rows.append({
            'persistence_cluster_id':f'P{int(lab):04d}',
            'centroid_x':cx,'centroid_y':cy,
            'persistence_detection_count':len(g),
            'persistence_dev_unique_dates':g.date.nunique(),
            'persistence_dev_unique_years':g.year.nunique(),
            'persistence_median_radial_dispersion_m':float(np.median(radial)),
            'persistence_max_radial_dispersion_m':float(np.max(radial)),
            'persistence_orientation_variability_deg':orient_var,
            'persistence_length_relative_iqr':length_iqr_rel,
            'persistence_width_relative_iqr':width_iqr_rel,
            'persistence_geometry_consistent':geometry_consistent,
            'member_candidate_ids':'|'.join(g.candidate_id.astype(str)),
        })

    c=pd.DataFrame(rows)
    if len(c):
        c['strong_static_cluster']=(
            (c.persistence_dev_unique_dates>=CFG['static_min_unique_dates'])
            & (c.persistence_dev_unique_years>=CFG['static_min_unique_years'])
            & (c.persistence_median_radial_dispersion_m<=CFG['static_median_dispersion_max_m'])
            & (c.persistence_max_radial_dispersion_m<=CFG['static_max_dispersion_max_m'])
            & c.persistence_geometry_consistent.fillna(False)
        )
    return c

def attach_persistence(all_df, clusters):
    out=add_xy(all_df)
    cols=[
        'persistence_cluster_id','persistence_dev_unique_dates','persistence_dev_unique_years',
        'persistence_detection_count','persistence_median_radial_dispersion_m',
        'persistence_max_radial_dispersion_m','persistence_orientation_variability_deg',
        'persistence_length_relative_iqr','persistence_width_relative_iqr',
        'persistence_geometry_consistent','strong_static_cluster',
        'distance_to_persistence_centroid_m'
    ]
    for c in cols:
        if c in ['strong_static_cluster','persistence_geometry_consistent']:
            out[c]=False
        elif c=='persistence_cluster_id':
            out[c]=None
        else:
            out[c]=np.nan

    if clusters is None or len(clusters)==0:
        return out.drop(columns=['utm_x','utm_y'])

    C=clusters[['centroid_x','centroid_y']].values
    P=out[['utm_x','utm_y']].values
    D=np.sqrt(((P[:,None,:]-C[None,:,:])**2).sum(axis=2))
    nearest=D.argmin(axis=1); mind=D.min(axis=1)

    for i,(j,d) in enumerate(zip(nearest,mind)):
        if d>CFG['persistence_eps_m']:
            continue
        c=clusters.iloc[j]
        for key in cols[:-1]:
            out.at[out.index[i],key]=c.get(key,np.nan)
        out.at[out.index[i],'distance_to_persistence_centroid_m']=float(d)

    return out.drop(columns=['utm_x','utm_y'])


In [ ]:
# STEP 22 - Build persistence from DEVELOPMENT ONLY
# Merge currently available evidence first.
evidence=fused.merge(spatial,on='candidate_id',how='left').merge(sar_features,on='candidate_id',how='left').merge(ais_assoc.drop(columns=['scene_id'],errors='ignore'),on='candidate_id',how='left')

# Strict leakage protection: persistence is fit from development rows only.
dev_evidence=evidence[evidence.experiment_set=='DEVELOPMENT'].copy()
assert not (dev_evidence.experiment_set=='HOLDOUT').any()

clusters=load_table('07_temporal/DEVELOPMENT_PERSISTENCE_CLUSTERS_V4_1')
if clusters is None or FORCE['temporal']:
    clusters=fit_persistence_clusters(dev_evidence)
    save_table(clusters,DIRS['temporal']/'DEVELOPMENT_PERSISTENCE_CLUSTERS_V4_1')

# Attach development-derived clusters to all currently processed candidates. Holdouts never create clusters.
temporal=attach_persistence(evidence,clusters)
save_table(temporal,DIRS['temporal']/'TEMPORAL_FEATURES_V4_1')
stage_complete('DEVELOPMENT-ONLY PERSISTENCE', input_rows=len(evidence), output_rows=len(temporal), checkpoint=DIRS['temporal'])

# Part J - Development-derived thresholds and conservative five-class classification

Thresholds are derived from development distributions where reasonable, but they are **not claimed to be accuracy-optimized** unless independent labels exist. Geometry and land-conflict limits remain explicit engineering constraints. The resulting JSON records the source of each threshold.

The final classifier is intentionally conservative near land/coast. A no-AIS detection is not automatically promoted to a vessel.

In [ ]:
# STEP 23 - Threshold derivation and classification functions

def finite_quantile(s,q,default):
    x=pd.to_numeric(s,errors='coerce').dropna()
    return float(x.quantile(q)) if len(x) else float(default)


def derive_thresholds(dev):
    return {
        'frozen': True,
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'source': 'DEVELOPMENT_ONLY; distribution-adaptive thresholds are provisional, not accuracy-optimized without gold-standard labels',
        'development_scene_ids': sorted(dev.scene_id.unique().tolist()),
        'high_detector_confidence': float(np.clip(finite_quantile(dev.confidence_max,0.65,0.55),0.45,0.68)),
        'strong_vv_contrast_db': float(np.clip(finite_quantile(dev.vv_target_background_contrast_db,0.55,4.0),2.5,6.5)),
        'strong_vv_robust_z': float(np.clip(finite_quantile(dev.vv_robust_contrast_z,0.55,3.0),2.0,5.0)),
        'strong_bbox_land_overlap': CFG['strong_bbox_land_overlap'],
        'strong_target_land_overlap': CFG['strong_target_land_overlap'],
        'ais_credible_qualities': ['HIGH','MODERATE'],
        'coastal_extra_evidence_required': True,
        'static_rules': {
            'min_unique_dates':CFG['static_min_unique_dates'],'min_unique_years':CFG['static_min_unique_years'],
            'median_dispersion_max_m':CFG['static_median_dispersion_max_m'],'max_dispersion_max_m':CFG['static_max_dispersion_max_m'],
        },
    }


def classify_candidate(r,t):
    reasons=[]
    target_land=pd.to_numeric(
        pd.Series([r.get('target_land_overlap_fraction',np.nan)]),
        errors='coerce'
    ).iloc[0]
    bbox_land=float(r.get('bbox_land_overlap_fraction',0) or 0)
    center_land=bool(r.get('center_on_land',False))
    coast_zone=str(r.get('coast_zone',''))

    strong_land=(
        center_land
        or bbox_land>=t['strong_bbox_land_overlap']
        or (np.isfinite(target_land) and target_land>=t['strong_target_land_overlap'])
    )
    moderate_land=(
        bbox_land>=CFG['moderate_bbox_land_overlap']
        or (np.isfinite(target_land) and target_land>=CFG['moderate_target_land_overlap'])
    )

    if strong_land:
        reasons.append('STRONG_LAND_CONFLICT')
    elif moderate_land:
        reasons.append('MODERATE_LAND_CONFLICT')

    coastal=coast_zone in ['LAND_OR_INTERSECTING','0_TO_300_M']
    if coastal:
        reasons.append('COASTAL_AMBIGUITY')

    credible_ais=(
        bool(r.get('ais_matched',False))
        and str(r.get('ais_match_quality','NONE')) in t['ais_credible_qualities']
    )
    if credible_ais:
        reasons += ['AIS_SPATIAL_MATCH','AIS_TEMPORAL_SUPPORT']

    consensus=bool(r.get('model_consensus',False))
    high_conf=float(r.get('confidence_max',0) or 0) >= t['high_detector_confidence']
    vv_contrast=float(
        r.get('vv_target_background_contrast_db',-999)
        if pd.notna(r.get('vv_target_background_contrast_db',np.nan)) else -999
    )
    vv_z=float(
        r.get('vv_robust_contrast_z',-999)
        if pd.notna(r.get('vv_robust_contrast_z',np.nan)) else -999
    )
    strong_contrast=(
        vv_contrast>=t['strong_vv_contrast_db']
        or vv_z>=t['strong_vv_robust_z']
    )
    geom=bool(r.get('geometry_plausible',False))

    if consensus: reasons.append('MODEL_CONSENSUS')
    if high_conf: reasons.append('HIGH_DETECTOR_CONFIDENCE')
    if strong_contrast: reasons.append('STRONG_SAR_CONTRAST')
    if geom: reasons.append('PLAUSIBLE_SAR_GEOMETRY')

    centroid_distance=pd.to_numeric(
        pd.Series([r.get('distance_to_persistence_centroid_m',np.nan)]),
        errors='coerce'
    ).iloc[0]
    static=(
        bool(r.get('strong_static_cluster',False))
        and np.isfinite(centroid_distance)
        and centroid_distance<=CFG['static_candidate_centroid_max_m']
    )
    if static:
        reasons += ['PERSISTENT_FIXED_TARGET','LOW_POSITIONAL_DISPERSION']

    # 1. A target centred on mapped land is never automatically rescued to vessel
    #    by a nearby AIS record. Near the shoreline it remains uncertain if AIS is
    #    strong, otherwise it is a probable false alarm.
    if center_land:
        if credible_ais and float(r.get('distance_to_coast_m',999999) or 999999) <= 75:
            return 'UNCERTAIN','LAND_AIS_CONFLICT',';'.join(dict.fromkeys(reasons)),'LOW'
        return 'PROBABLE_FALSE_ALARM','CENTER_ON_LAND',';'.join(dict.fromkeys(reasons)),'HIGH'

    # 2. Strong target/bbox land overlap without credible AIS support.
    if strong_land and not credible_ais:
        return 'PROBABLE_FALSE_ALARM','LAND_OVERLAP',';'.join(dict.fromkeys(reasons)),'HIGH'

    # 3. A strong land conflict plus AIS is still conflicting evidence, not an
    #    automatic vessel label.
    if strong_land and credible_ais:
        return 'UNCERTAIN','LAND_AIS_CONFLICT',';'.join(dict.fromkeys(reasons)),'LOW'

    # 4. Strong development-derived fixed-target evidence.
    if static and not credible_ais:
        return 'STATIC_MARITIME_OBJECT','PERSISTENT_FIXED_TARGET',';'.join(dict.fromkeys(reasons)),'HIGH'

    # 5. Credible AIS association can support a vessel only when land conflict
    #    is not dominant.
    if credible_ais:
        level='HIGH' if str(r.get('ais_match_quality'))=='HIGH' and not moderate_land else 'MODERATE'
        primary='COASTAL_AIS_SUPPORTED' if moderate_land or coastal else 'AIS_SPATIAL_MATCH'
        return 'AIS_MATCHED_VESSEL',primary,';'.join(dict.fromkeys(reasons)),level

    # 6. Non-AIS vessel promotion.
    # Offshore: 3 evidence units are enough.
    # Nearshore: require the three physically meaningful units together:
    # consensus + strong SAR contrast + plausible geometry. High confidence is
    # supportive but is not mandatory, which avoids rejecting genuine coastal
    # ships solely because clutter lowered detector confidence.
    low_land=(
        bbox_land < CFG['coastal_low_land_bbox_max']
        and (not np.isfinite(target_land) or target_land < CFG['coastal_low_land_target_max'])
    )

    if coastal:
        if low_land and consensus and strong_contrast and geom and not static:
            return 'NON_AIS_LIKELY_VESSEL','COASTAL_MULTI_EVIDENCE',';'.join(dict.fromkeys(reasons)),'MODERATE'
    else:
        evidence_units=int(consensus)+int(high_conf)+int(strong_contrast)+int(geom)
        if low_land and evidence_units>=3 and strong_contrast and geom and not static:
            return 'NON_AIS_LIKELY_VESSEL','STRONG_SAR_CONTRAST',';'.join(dict.fromkeys(reasons)),'MODERATE'

    # 7. Moderate land conflict is deliberately not promoted to ship without AIS.
    if moderate_land and not credible_ais:
        if (not strong_contrast) or (not geom):
            return 'PROBABLE_FALSE_ALARM','MODERATE_LAND_WEAK_SHIP_EVIDENCE',';'.join(dict.fromkeys(reasons)),'MODERATE'
        return 'UNCERTAIN','MODERATE_LAND_CONFLICT',';'.join(dict.fromkeys(reasons)),'LOW'

    weak_model=(not consensus and not high_conf)
    weak_sar=(not strong_contrast)
    bad_geom=(not geom and str(r.get('dimension_quality','')).startswith('OK'))

    if (weak_model and weak_sar) or (bad_geom and weak_sar):
        if weak_model: reasons.append('SINGLE_MODEL_LOW_CONFIDENCE')
        if weak_sar: reasons.append('WEAK_SAR_CONTRAST')
        if bad_geom: reasons.append('IMPLAUSIBLE_GEOMETRY')
        primary=reasons[-1] if reasons else 'WEAK_SAR_CONTRAST'
        return 'PROBABLE_FALSE_ALARM',primary,';'.join(dict.fromkeys(reasons)),'MODERATE'

    reasons.append('CONFLICTING_EVIDENCE')
    return 'UNCERTAIN','CONFLICTING_EVIDENCE',';'.join(dict.fromkeys(reasons)),'LOW'

def apply_classifier(df, thresholds):
    rows=[]
    for _,r in df.iterrows():
        c,p,s,l=classify_candidate(r,thresholds)
        rows.append((c,p,s,l))
    out=df.copy()
    out[['final_class','primary_reason_code','secondary_reason_codes','classification_confidence_level']]=pd.DataFrame(rows,index=out.index)
    assert set(out.final_class.unique()).issubset(set(CLASS_ORDER))
    return out

In [ ]:
# STEP 24 - Freeze development thresholds, then classify available scenes
# Threshold derivation can use DEVELOPMENT only.
dev_temporal=temporal[temporal.experiment_set=='DEVELOPMENT'].copy()
assert set(dev_temporal.experiment_set.unique()) <= {'DEVELOPMENT'}

frozen_path=DIRS['classification']/'FROZEN_THRESHOLDS_V4_1.json'
read_frozen=read_path('08_classification/FROZEN_THRESHOLDS_V4_1.json')
if read_frozen.exists() and not FORCE['classification']:
    thresholds=load_json(read_frozen)
    print('Loaded frozen thresholds:',read_frozen)
else:
    thresholds=derive_thresholds(dev_temporal)
    # Strong leakage assertion.
    holdout_ids=set(scene_catalog.loc[scene_catalog.experiment_set=='HOLDOUT','scene_id'].astype(str))
    assert holdout_ids.isdisjoint(set(thresholds['development_scene_ids']))
    save_json(frozen_path,thresholds)
    print('Development thresholds frozen:',frozen_path)

classified=apply_classifier(temporal,thresholds)
save_table(classified,DIRS['classification']/'FINAL_CLASSIFIED_SAR_CANDIDATES')
save_table(classified[classified.experiment_set=='DEVELOPMENT'],DIRS['classification']/'DEVELOPMENT_CLASSIFIED')
if (classified.experiment_set=='HOLDOUT').any():
    save_table(classified[classified.experiment_set=='HOLDOUT'],DIRS['classification']/'HOLDOUT_CLASSIFIED')

summary=(classified.groupby(['experiment_set','final_class']).size().rename('count').reset_index())
summary['final_class']=pd.Categorical(summary.final_class,categories=CLASS_ORDER,ordered=True)
summary=summary.sort_values(['experiment_set','final_class'])
summary.to_csv(DIRS['classification']/'CLASSIFICATION_SUMMARY.csv',index=False)
print(summary.to_string(index=False))
stage_complete('FIVE-CLASS CLASSIFICATION', input_rows=len(temporal), output_rows=len(classified), checkpoint=DIRS['classification'])

In [ ]:
# STEP 24B - FINAL CLASSIFICATION MAPS + TARGET-PROPERTY QA
#
# These outputs are always saved so the compact final ZIP contains actual
# visual products, not CSV-only results.

from matplotlib.patches import Rectangle, Patch

CLASS_COLORS = {
    'AIS_MATCHED_VESSEL': '#00A651',
    'NON_AIS_LIKELY_VESSEL': '#00B7EB',
    'STATIC_MARITIME_OBJECT': '#FF9F1C',
    'PROBABLE_FALSE_ALARM': '#D62728',
    'UNCERTAIN': '#9467BD',
}

MAP_ROOT = DIRS['figures']/'final_classification_maps'
SAR_MAP_DIR = MAP_ROOT/'sar_overlays'
GEO_MAP_DIR = MAP_ROOT/'geographic_maps'
VESSEL_MAP_DIR = MAP_ROOT/'vessel_only'
PROP_DIR = DIRS['figures']/'target_property_plots'
for p in [SAR_MAP_DIR,GEO_MAP_DIR,VESSEL_MAP_DIR,PROP_DIR]:
    p.mkdir(parents=True,exist_ok=True)

def draw_class_boxes(ax, g, cache, classes=None, linewidth=0.8):
    classes = classes or CLASS_ORDER
    for cls in classes:
        q=g[g.final_class==cls]
        for r in q.itertuples(index=False):
            x=float(r.global_x1-cache['vv_x_min'])
            y=float(r.global_y1-cache['vv_y_min'])
            w=float(r.global_x2-r.global_x1)
            h=float(r.global_y2-r.global_y1)
            ax.add_patch(Rectangle(
                (x,y),w,h,
                fill=False,
                edgecolor=CLASS_COLORS[cls],
                linewidth=linewidth,
                alpha=0.95
            ))

def save_scene_classification_products(scene_id, g):
    cache=load_scene_cache(scene_id)
    image=cache['model_image']

    # A. SAR overlay with all five final classes
    fig,ax=plt.subplots(figsize=(18,10))
    ax.imshow(image,cmap='gray',vmin=0,vmax=255)
    draw_class_boxes(ax,g,cache)
    handles=[
        Patch(facecolor='none',edgecolor=CLASS_COLORS[c],
              label=f"{c} ({int((g.final_class==c).sum())})")
        for c in CLASS_ORDER
    ]
    ax.legend(handles=handles,loc='lower right',fontsize=8,framealpha=0.9)
    ax.set_title(f'{scene_id} - final object-level classification')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(SAR_MAP_DIR/f'{scene_id}_FINAL_CLASSIFICATION_OVERLAY.png',
                dpi=220,bbox_inches='tight')
    plt.close(fig)

    # B. Vessel-only overlay
    vessel_classes=['AIS_MATCHED_VESSEL','NON_AIS_LIKELY_VESSEL']
    fig,ax=plt.subplots(figsize=(18,10))
    ax.imshow(image,cmap='gray',vmin=0,vmax=255)
    draw_class_boxes(ax,g,cache,classes=vessel_classes,linewidth=1.0)
    handles=[
        Patch(facecolor='none',edgecolor=CLASS_COLORS[c],
              label=f"{c} ({int((g.final_class==c).sum())})")
        for c in vessel_classes
    ]
    ax.legend(handles=handles,loc='lower right',fontsize=8,framealpha=0.9)
    ax.set_title(f'{scene_id} - vessel-only result')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(VESSEL_MAP_DIR/f'{scene_id}_VESSEL_ONLY_OVERLAY.png',
                dpi=220,bbox_inches='tight')
    plt.close(fig)

    # C. Geographic classification map with shoreline
    fig,ax=plt.subplots(figsize=(12,7))
    try:
        shore.boundary.plot(ax=ax,color='black',linewidth=0.6)
    except Exception:
        pass
    for cls in CLASS_ORDER:
        q=g[g.final_class==cls]
        if len(q):
            ax.scatter(q.longitude,q.latitude,s=10,
                       c=CLASS_COLORS[cls],label=f'{cls} ({len(q)})',
                       alpha=0.75)
    ax.set_xlim(CFG['lon_min'],CFG['lon_max'])
    ax.set_ylim(CFG['lat_min'],CFG['lat_max'])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{scene_id} - geographic final classification')
    ax.legend(loc='best',fontsize=7)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(GEO_MAP_DIR/f'{scene_id}_GEOGRAPHIC_CLASSIFICATION_MAP.png',
                dpi=220,bbox_inches='tight')
    plt.close(fig)

if MAKE_FINAL_CLASSIFICATION_MAPS:
    for sid,g in classified.groupby('scene_id',sort=True):
        save_scene_classification_products(sid,g)
    print("Final classification maps saved:",MAP_ROOT)

# Object-level GeoJSON is the correct GIS product for this detector.
# This is not a per-pixel semantic-classification raster.
if EXPORT_FINAL_GEOJSON and len(classified):
    geo_cols=[c for c in [
        'scene_id','candidate_id','scene_datetime','experiment_set',
        'final_class','classification_confidence_level','primary_reason_code',
        'longitude','latitude','confidence_max','model_consensus',
        'distance_to_coast_m','coast_zone','bbox_land_overlap_fraction',
        'target_land_overlap_fraction','ais_matched','ais_match_quality',
        'nearest_ais_distance_m','sar_apparent_length_m','sar_apparent_width_m',
        'sar_aspect_ratio','sar_orientation_deg','vv_target_background_contrast_db'
    ] if c in classified.columns]
    gdf=gpd.GeoDataFrame(
        classified[geo_cols].copy(),
        geometry=gpd.points_from_xy(classified.longitude,classified.latitude),
        crs='EPSG:4326'
    )
    gdf.to_file(DIRS['classification']/'FINAL_CLASSIFIED_SAR_CANDIDATES.geojson',
                driver='GeoJSON')
    print("GeoJSON saved:",DIRS['classification']/'FINAL_CLASSIFIED_SAR_CANDIDATES.geojson')

# ---------------------------
# Target-property summary
# ---------------------------
property_cols=[
    'sar_signature_area_m2','sar_apparent_length_m','sar_apparent_width_m',
    'sar_aspect_ratio','sar_orientation_deg','sar_compactness','sar_fill_ratio',
    'vv_target_background_contrast_db','vh_target_background_contrast_db',
    'vv_robust_contrast_z','target_land_overlap_fraction'
]
available_props=[c for c in property_cols if c in classified.columns]

if available_props:
    prop_summary=classified.groupby('final_class')[available_props].agg(
        ['count','median','mean','std']
    )
    prop_summary.to_csv(DIRS['tables']/'TARGET_PROPERTY_SUMMARY_BY_CLASS.csv')

if MAKE_TARGET_PROPERTY_PLOTS and len(classified):
    # 1. Apparent length by final class
    data=[]; labels=[]
    for cls in CLASS_ORDER:
        x=pd.to_numeric(classified.loc[classified.final_class==cls,'sar_apparent_length_m'],
                        errors='coerce').dropna()
        if len(x):
            data.append(x.values); labels.append(cls)
    if data:
        fig,ax=plt.subplots(figsize=(12,6))
        ax.boxplot(data,tick_labels=labels,showfliers=False)
        ax.set_ylabel('Apparent SAR length (m)')
        ax.set_title('Apparent SAR-signature length by final class')
        ax.tick_params(axis='x',rotation=25)
        fig.tight_layout()
        fig.savefig(PROP_DIR/'01_APPARENT_LENGTH_BY_CLASS.png',dpi=220,bbox_inches='tight')
        plt.close(fig)

    # 2. Apparent width by final class
    data=[]; labels=[]
    for cls in CLASS_ORDER:
        x=pd.to_numeric(classified.loc[classified.final_class==cls,'sar_apparent_width_m'],
                        errors='coerce').dropna()
        if len(x):
            data.append(x.values); labels.append(cls)
    if data:
        fig,ax=plt.subplots(figsize=(12,6))
        ax.boxplot(data,tick_labels=labels,showfliers=False)
        ax.set_ylabel('Apparent SAR width (m)')
        ax.set_title('Apparent SAR-signature width by final class')
        ax.tick_params(axis='x',rotation=25)
        fig.tight_layout()
        fig.savefig(PROP_DIR/'02_APPARENT_WIDTH_BY_CLASS.png',dpi=220,bbox_inches='tight')
        plt.close(fig)

    # 3. Length-width scatter
    fig,ax=plt.subplots(figsize=(8,7))
    for cls in CLASS_ORDER:
        q=classified[classified.final_class==cls]
        x=pd.to_numeric(q.sar_apparent_length_m,errors='coerce')
        y=pd.to_numeric(q.sar_apparent_width_m,errors='coerce')
        m=np.isfinite(x)&np.isfinite(y)
        if m.any():
            ax.scatter(x[m],y[m],s=8,alpha=0.35,c=CLASS_COLORS[cls],label=cls)
    ax.set_xlabel('Apparent SAR length (m)')
    ax.set_ylabel('Apparent SAR width (m)')
    ax.set_title('Apparent target geometry')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(PROP_DIR/'03_LENGTH_VS_WIDTH.png',dpi=220,bbox_inches='tight')
    plt.close(fig)

    # 4. SAR apparent length versus AIS vessel length
    if 'matched_vessel_length_m' in classified.columns:
        q=classified[
            classified.ais_matched.fillna(False)
            & classified.matched_vessel_length_m.notna()
            & classified.sar_apparent_length_m.notna()
        ].copy()
        q['matched_vessel_length_m']=pd.to_numeric(q.matched_vessel_length_m,errors='coerce')
        q['sar_apparent_length_m']=pd.to_numeric(q.sar_apparent_length_m,errors='coerce')
        q=q[np.isfinite(q.matched_vessel_length_m)&np.isfinite(q.sar_apparent_length_m)]
        if len(q):
            fig,ax=plt.subplots(figsize=(7,7))
            ax.scatter(q.matched_vessel_length_m,q.sar_apparent_length_m,
                       s=10,alpha=0.35)
            mx=float(np.nanpercentile(
                np.r_[q.matched_vessel_length_m.values,q.sar_apparent_length_m.values],99
            ))
            mx=max(mx,50)
            ax.plot([0,mx],[0,mx],'--',linewidth=1,label='1:1 reference')
            ax.set_xlim(0,mx); ax.set_ylim(0,mx)
            ax.set_xlabel('AIS vessel length (m)')
            ax.set_ylabel('Apparent SAR length (m)')
            ax.set_title('Geometry QA: AIS length vs apparent SAR length')
            ax.legend()
            ax.grid(alpha=0.2)
            fig.tight_layout()
            fig.savefig(PROP_DIR/'04_AIS_LENGTH_VS_SAR_APPARENT_LENGTH.png',
                        dpi=220,bbox_inches='tight')
            plt.close(fig)

            q[['scene_id','candidate_id','matched_vessel_length_m',
               'sar_apparent_length_m','sar_apparent_width_m',
               'sar_orientation_deg','dimension_quality']].to_csv(
                DIRS['tables']/'AIS_VS_SAR_GEOMETRY_QA.csv',index=False
            )

    # 5. VV contrast by final class
    data=[]; labels=[]
    for cls in CLASS_ORDER:
        x=pd.to_numeric(
            classified.loc[classified.final_class==cls,'vv_target_background_contrast_db'],
            errors='coerce'
        ).dropna()
        if len(x):
            data.append(x.values); labels.append(cls)
    if data:
        fig,ax=plt.subplots(figsize=(12,6))
        ax.boxplot(data,tick_labels=labels,showfliers=False)
        ax.set_ylabel('VV target-background contrast (dB)')
        ax.set_title('SAR contrast by final class')
        ax.tick_params(axis='x',rotation=25)
        fig.tight_layout()
        fig.savefig(PROP_DIR/'05_VV_CONTRAST_BY_CLASS.png',dpi=220,bbox_inches='tight')
        plt.close(fig)

    # 6. Segmentation quality counts
    if 'dimension_quality' in classified.columns:
        qc=classified.dimension_quality.fillna('MISSING').value_counts()
        fig,ax=plt.subplots(figsize=(9,5))
        ax.bar(qc.index.astype(str),qc.values)
        ax.set_ylabel('Candidates')
        ax.set_title('Target-segmentation quality')
        ax.tick_params(axis='x',rotation=30)
        fig.tight_layout()
        fig.savefig(PROP_DIR/'06_SEGMENTATION_QUALITY_COUNTS.png',
                    dpi=220,bbox_inches='tight')
        plt.close(fig)

    print("Target-property plots saved:",PROP_DIR)


# Part K - Better validation design

This project has 15 development scenes and 3 locked holdouts.

The notebook performs **leave-one-scene-out (LOSO) rule-stability analysis** inside the development set. For each fold, thresholds and persistence clusters are built from the other development scenes, then applied to the omitted scene. Without independent manual ground truth, this is a stability analysis, not formal precision/recall.

A manual annotation template is also generated. Formal accuracy metrics are calculated only if you later fill a reviewed label file.

In [ ]:
# STEP 25 - Manual validation template and LOSO development stability
manual_template=DIRS['validation']/'manual_annotation_template.csv'
if not manual_template.exists():
    tmpl=classified[['scene_id','candidate_id','longitude','latitude','experiment_set']].copy()
    tmpl['reference_source']='CANDIDATE_REVIEW'
    tmpl['manual_label']=''
    tmpl['review_confidence']=''
    tmpl['ais_present']=tmpl.candidate_id.map(dict(zip(classified.candidate_id,classified.ais_matched))).fillna(False)
    tmpl['ais_vessel_id']=tmpl.candidate_id.map(dict(zip(classified.candidate_id,classified.matched_ais_id)))
    tmpl['expected_sar_visible']=''
    tmpl['review_notes']=''
    tmpl.to_csv(manual_template,index=False)
    print('Created:',manual_template)


def loso_development_stability(dev):
    fold_rows=[]; thr_rows=[]
    for held_sid in sorted(dev.scene_id.unique()):
        train=dev[dev.scene_id!=held_sid].copy(); test=dev[dev.scene_id==held_sid].copy()
        if len(train)==0 or len(test)==0: continue
        t=derive_thresholds(train)
        # Refit persistence using the 14 training scenes only and attach the held-out development scene.
        cl=fit_persistence_clusters(train)
        test_base=test.drop(columns=[c for c in test.columns if c.startswith('persistence_') or c in ['strong_static_cluster','distance_to_persistence_centroid_m']],errors='ignore')
        test_temp=attach_persistence(test_base,cl)
        pred=apply_classifier(test_temp,t)
        counts=pred.final_class.value_counts()
        row={'held_out_scene_id':held_sid,'n_candidates':len(pred),'n_static_matches':int(pred.strong_static_cluster.fillna(False).sum()),
             'vessel_like_fraction':float(pred.final_class.isin(['AIS_MATCHED_VESSEL','NON_AIS_LIKELY_VESSEL']).mean())}
        for c in CLASS_ORDER: row[f'n_{c}']=int(counts.get(c,0))
        fold_rows.append(row)
        thr_rows.append({'held_out_scene_id':held_sid,'high_detector_confidence':t['high_detector_confidence'],
                         'strong_vv_contrast_db':t['strong_vv_contrast_db'],'strong_vv_robust_z':t['strong_vv_robust_z']})
    return pd.DataFrame(fold_rows),pd.DataFrame(thr_rows)

if RUN['validation']:
    loso,thr_stability=loso_development_stability(dev_temporal)
    loso.to_csv(DIRS['validation']/'LOSO_RESULTS.csv',index=False)
    thr_stability.to_csv(DIRS['validation']/'THRESHOLD_STABILITY.csv',index=False)
    print('LOSO scenes:',len(loso))
    if len(thr_stability):
        print(thr_stability.describe().loc[['mean','std','min','max']].to_string())

In [ ]:
# STEP 26 - Optional formal metrics from manually reviewed labels
# To enable, attach or copy a file named MANUAL_GOLD_STANDARD.csv into 09_validation or /kaggle/input.

def find_manual_gold():
    local=read_path('09_validation/MANUAL_GOLD_STANDARD.csv')
    if local.exists(): return local
    files=list(KAGGLE_INPUT.rglob('MANUAL_GOLD_STANDARD.csv')) if KAGGLE_INPUT.exists() else []
    return files[0] if files else None

manual_gold=find_manual_gold()
formal_metrics=[]
if manual_gold is None:
    print('No MANUAL_GOLD_STANDARD.csv found. Formal precision/recall are NOT reported.')
else:
    gold=pd.read_csv(manual_gold)
    # Map candidate review labels to the five operational classes only when unambiguous.
    mapping={
        'VESSEL':'VESSEL',
        'STATIC_STRUCTURE':'NON_VESSEL',
        'LAND_ISLAND_COAST':'NON_VESSEL',
        'CLUTTER_FALSE_ALARM':'NON_VESSEL',
    }
    evaldf=classified.merge(gold[['candidate_id','manual_label','review_confidence']],on='candidate_id',how='inner')
    evaldf['truth_binary']=evaldf.manual_label.map(mapping)
    evaldf=evaldf[evaldf.truth_binary.notna()].copy()
    evaldf['pred_binary']=np.select([
        evaldf.final_class.isin(['AIS_MATCHED_VESSEL','NON_AIS_LIKELY_VESSEL']),
        evaldf.final_class.isin(['STATIC_MARITIME_OBJECT','PROBABLE_FALSE_ALARM'])
    ],['VESSEL','NON_VESSEL'],default='ABSTAIN')
    if len(evaldf):
        covered=evaldf[evaldf.pred_binary!='ABSTAIN'].copy()
        abstain_rate=float((evaldf.pred_binary=='ABSTAIN').mean())
        if len(covered):
            p,r,f,_=precision_recall_fscore_support(covered.truth_binary,covered.pred_binary,average='binary',pos_label='VESSEL',zero_division=0)
        else:
            p=r=f=np.nan
        formal_metrics=[{'n_reviewed':len(evaldf),'n_scored_non_abstain':len(covered),'abstain_rate':abstain_rate,
                         'precision_vessel_on_scored':p,'recall_vessel_on_scored':r,'f1_vessel_on_scored':f}]
        pd.DataFrame(formal_metrics).to_csv(DIRS['validation']/'FORMAL_VALIDATION_METRICS.csv',index=False)
        print(pd.DataFrame(formal_metrics).to_string(index=False))

In [ ]:
# STEP 27 - Locked holdout evaluation summary
# This is formal only if independent labels exist; otherwise it is evidence-based holdout evaluation.
hold=classified[classified.experiment_set=='HOLDOUT'].copy()
if len(hold)==0:
    print('Holdout has not been processed yet. This is correct for the first development run.')
    print('After thresholds are frozen, set PROCESS_HOLDOUT=True and rerun from scene processing onward.')
else:
    assert read_path('08_classification/FROZEN_THRESHOLDS_V4_1.json').exists()
    hsum=(hold.groupby('final_class').size().reindex(CLASS_ORDER,fill_value=0).rename('count').reset_index())
    hsum['evaluation_type']='evidence-based holdout evaluation' if manual_gold is None else 'label-supported holdout evaluation where eligible'
    hsum.to_csv(DIRS['validation']/'HOLDOUT_EVALUATION_SUMMARY.csv',index=False)
    hold[['scene_id','candidate_id','final_class','classification_confidence_level','primary_reason_code']].to_csv(DIRS['validation']/'HOLDOUT_SCENE_RESULTS.csv',index=False)
    print(hsum.to_string(index=False))

# Part L - Hard-negative mining

High-confidence island/coast/static false alarms are scientifically useful training data for a later E2 model. This notebook only catalogs them. It does **not** automatically retrain E0/E1.

In [ ]:
# STEP 28 - Build compact hard-negative catalogue
if RUN['hard_negatives']:
    hn=classified[
        (classified.final_class=='PROBABLE_FALSE_ALARM')
        & (
            (classified.confidence_max>=thresholds['high_detector_confidence'])
            | (classified.bbox_land_overlap_fraction>=0.30)
            | (classified.strong_static_cluster.fillna(False))
        )
    ].copy()
    def hn_category(r):
        if r.get('bbox_land_overlap_fraction',0)>=0.30 or (pd.notna(r.get('target_land_overlap_fraction')) and r.get('target_land_overlap_fraction')>=0.30):
            return 'ISLAND_LAND_COAST_RESPONSE'
        if bool(r.get('strong_static_cluster',False)):
            return 'PERSISTENT_FIXED_RESPONSE'
        if r.get('coast_zone') in ['LAND_OR_INTERSECTING','0_TO_300_M']:
            return 'NEARSHORE_CLUTTER'
        return 'OTHER_HIGH_CONF_FALSE_ALARM'
    if len(hn): hn['hard_negative_category']=hn.apply(hn_category,axis=1)
    keep=[c for c in ['scene_id','candidate_id','longitude','latitude','confidence_max','coast_zone','bbox_land_overlap_fraction',
                      'target_land_overlap_fraction','vv_target_background_contrast_db','sar_apparent_length_m','sar_apparent_width_m',
                      'final_class','primary_reason_code','hard_negative_category'] if c in hn.columns]
    hn[keep].to_csv(DIRS['hard_negatives']/'HARD_NEGATIVE_CATALOG.csv',index=False)
    print('Hard-negative candidates:',len(hn))

# Part M - Compact final outputs and Kaggle persistence

The final export intentionally avoids dozens of redundant files. The main research deliverables are:

- locked scene catalogue,
- final five-class candidate table,
- main vessel-only result table,
- target properties,
- AIS association and timestamps,
- classification summary,
- development-derived thresholds,
- validation/holdout summary,
- hard-negative catalogue,
- run manifest.

Detailed raw detections remain in checkpoint folders for reproducibility but are not copied into the compact final-results ZIP.

In [ ]:
# STEP 29 - Create main result tables
if RUN['export']:
    final_all=classified.copy()
    main_vessels=final_all[final_all.final_class.isin(['AIS_MATCHED_VESSEL','NON_AIS_LIKELY_VESSEL'])].copy()

    main_cols=[c for c in [
        'scene_id','candidate_id','scene_datetime','year','experiment_set','longitude','latitude',
        'final_class','classification_confidence_level','primary_reason_code','secondary_reason_codes',
        'E0_detected','E1_detected','model_consensus','E0_confidence','E1_confidence','confidence_max',
        'center_on_land','bbox_land_overlap_fraction','target_land_overlap_fraction','distance_to_coast_m','coast_zone',
        'ais_matched','matched_ais_id','matched_vessel_name','matched_vessel_length_m','nearest_ais_distance_m',
        'sar_ais_time_difference_min','ais_time_mode','ais_match_quality','ais_before_time','ais_after_time','ais_source_time',
        'vv_target_median_db','vh_target_median_db','vv_background_median_db','vh_background_median_db',
        'vv_target_background_contrast_db','vh_target_background_contrast_db','vv_robust_contrast_z',
        'sar_signature_area_m2','sar_apparent_length_m','sar_apparent_width_m','sar_aspect_ratio','sar_orientation_deg',
        'sar_compactness','sar_fill_ratio','dimension_quality',
        'persistence_cluster_id','persistence_dev_unique_dates','persistence_dev_unique_years',
        'persistence_median_radial_dispersion_m','persistence_max_radial_dispersion_m','strong_static_cluster',
    ] if c in final_all.columns]

    final_all[main_cols].to_csv(DIRS['final']/'FINAL_CLASSIFIED_SAR_CANDIDATES.csv',index=False)
    final_all[main_cols].to_parquet(DIRS['final']/'FINAL_CLASSIFIED_SAR_CANDIDATES.parquet',index=False)
    main_vessels[main_cols].to_csv(DIRS['final']/'MAIN_VESSEL_RESULTS.csv',index=False)

    target_cols=[c for c in ['scene_id','candidate_id','longitude','latitude','vv_target_median_db','vh_target_median_db',
                              'vv_background_median_db','vh_background_median_db','vv_target_background_contrast_db',
                              'vh_target_background_contrast_db','vv_robust_contrast_z','sar_signature_area_m2',
                              'sar_apparent_length_m','sar_apparent_width_m','sar_aspect_ratio','sar_orientation_deg',
                              'sar_compactness','sar_fill_ratio','dimension_quality'] if c in final_all.columns]
    final_all[target_cols].to_csv(DIRS['final']/'TARGET_PROPERTIES.csv',index=False)

    ais_cols=[c for c in ['scene_id','candidate_id','scene_datetime','ais_matched','matched_ais_id','matched_vessel_name',
                           'nearest_ais_distance_m','sar_ais_time_difference_min','ais_time_mode','ais_match_quality',
                           'ais_before_time','ais_after_time','ais_source_time'] if c in final_all.columns]
    final_all[ais_cols].to_csv(DIRS['final']/'AIS_ASSOCIATION_AND_TIMESTAMPS.csv',index=False)

    summary.to_csv(DIRS['final']/'CLASSIFICATION_SUMMARY.csv',index=False)
    scene_catalog.to_csv(DIRS['final']/'LOCKED_SCENE_CATALOG.csv',index=False)
    shutil.copy2(read_path('08_classification/FROZEN_THRESHOLDS_V4_1.json'),DIRS['final']/'FROZEN_THRESHOLDS_V4_1.json')
    for src_name in ['LOSO_RESULTS.csv','THRESHOLD_STABILITY.csv','HOLDOUT_EVALUATION_SUMMARY.csv','FORMAL_VALIDATION_METRICS.csv']:
        src=DIRS['validation']/src_name
        if src.exists(): shutil.copy2(src,DIRS['final']/src_name)
    hn=DIRS['hard_negatives']/'HARD_NEGATIVE_CATALOG.csv'
    if hn.exists(): shutil.copy2(hn,DIRS['final']/hn.name)

    print('Final classified candidates:',len(final_all))
    print('Main vessel results:',len(main_vessels))
    print('\nMain vessel class counts:')
    print(main_vessels.final_class.value_counts().to_string())

# V4.1 - include actual visual outputs and GIS deliverables in FINAL_EXPORT.
if RUN['export']:
    final_fig_dir=DIRS['final']/'FIGURES'
    final_fig_dir.mkdir(parents=True,exist_ok=True)

    for source_dir,subname in [
        (DIRS['figures']/'final_classification_maps','classification_maps'),
        (DIRS['figures']/'target_property_plots','target_property_plots'),
        (DIRS['spatial']/'diagnostic_plots','spatial_diagnostics'),
    ]:
        if source_dir.exists():
            shutil.copytree(source_dir,final_fig_dir/subname,dirs_exist_ok=True)

    geojson_src=DIRS['classification']/'FINAL_CLASSIFIED_SAR_CANDIDATES.geojson'
    if geojson_src.exists():
        shutil.copy2(geojson_src,DIRS['final']/geojson_src.name)

    ais_audit_src=DIRS['validation']/'AIS_CENTERED_DETECTOR_AUDIT.csv'
    if ais_audit_src.exists():
        shutil.copy2(ais_audit_src,DIRS['final']/ais_audit_src.name)

    for extra in [
        DIRS['tables']/'TARGET_PROPERTY_SUMMARY_BY_CLASS.csv',
        DIRS['tables']/'AIS_VS_SAR_GEOMETRY_QA.csv',
    ]:
        if extra.exists():
            shutil.copy2(extra,DIRS['final']/extra.name)

    print("V4.1 final figures copied into:",final_fig_dir)


In [ ]:
# STEP 30 - Run manifest, compact final ZIP, and restart-critical checkpoint dataset
if RUN['export']:
    manifest={
        'run_timestamp_utc':datetime.now(timezone.utc).isoformat(),
        'python_version':sys.version,
        'platform':platform.platform(),
        'torch_version':torch.__version__,
        'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'model_paths':{k:str(v) for k,v in MODEL_PATHS.items()},
        'model_sha256':{k:file_sha256(v) for k,v in MODEL_PATHS.items()},
        'scene_catalog_sha256':file_sha256(DIRS['scene_catalog']/'LOCKED_SCENE_CATALOG.csv'),
        'n_scene_catalog':len(scene_catalog),
        'n_development':int((scene_catalog.experiment_set=='DEVELOPMENT').sum()),
        'n_holdout':int((scene_catalog.experiment_set=='HOLDOUT').sum()),
        'process_holdout_this_run':PROCESS_HOLDOUT,
        'debug_mode':DEBUG_MODE,
        'thresholds':thresholds,
        'final_candidate_rows':len(classified),
        'main_vessel_rows':int(classified.final_class.isin(['AIS_MATCHED_VESSEL','NON_AIS_LIKELY_VESSEL']).sum()),
    }
    save_json(DIRS['manifests']/'RUN_MANIFEST.json',manifest)
    shutil.copy2(DIRS['manifests']/'RUN_MANIFEST.json',DIRS['final']/'RUN_MANIFEST.json')

    final_zip=WORK_ROOT.parent/'SAR_Ship_Project_V4_1_FINAL_RESULTS.zip'
    if final_zip.exists(): final_zip.unlink()
    with zipfile.ZipFile(final_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in DIRS['final'].rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(DIRS['final']))

    checkpoint_dir=WORK_ROOT.parent/'SAR_Ship_Project_V4_1_CHECKPOINT_DATASET'
    if checkpoint_dir.exists(): shutil.rmtree(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True)
    # These are the expensive/restart-critical folders. Copy what exists locally.
    for rel in ['00_scene_catalog','01_scene_cache','02_raw_detections','03_candidate_fusion','06_ais','14_manifests']:
        src=WORK_ROOT/rel
        if src.exists(): shutil.copytree(src,checkpoint_dir/rel,dirs_exist_ok=True)
    # Include cheap downstream outputs too if available.
    for rel in ['04_spatial_context','05_sar_features','07_temporal','08_classification','09_validation']:
        src=WORK_ROOT/rel
        if src.exists(): shutil.copytree(src,checkpoint_dir/rel,dirs_exist_ok=True)

    persist_zip=WORK_ROOT.parent/'SAR_Ship_Project_V4_1_PERSIST_ME.zip'
    if persist_zip.exists(): persist_zip.unlink()
    with zipfile.ZipFile(persist_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for p in checkpoint_dir.rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(checkpoint_dir))

    print('FINAL RESULTS ZIP:',final_zip)
    print('CHECKPOINT DATASET:',checkpoint_dir)
    print('PERSIST ZIP:',persist_zip)

## Final Kaggle persistence instructions

After an expensive successful run:

1. Click **Save Version** in Kaggle.
2. Make sure notebook outputs are included.
3. Preserve `SAR_Ship_Project_V4_1_CHECKPOINT_DATASET/` as a Kaggle Dataset or attach the saved notebook output to the next notebook.
4. On the next session, attach that checkpoint dataset under `/kaggle/input`.
5. The notebook will auto-detect the previous `LOCKED_SCENE_CATALOG.csv` and reuse available caches/checkpoints.

For the first scientific run, keep `PROCESS_HOLDOUT = False`. After `08_classification/FROZEN_THRESHOLDS_V4_1.json` is saved and you have saved a Kaggle Version, set `PROCESS_HOLDOUT = True`. Do not alter classification thresholds after looking at holdout results.

In [ ]:
print('='*96)
print('RUN COMPLETE')
print('='*96)
print('Core final files:', DIRS['final'])
for p in sorted(DIRS['final'].glob('*')):
    if p.is_file(): print(' -',p.name)
print('\nIMPORTANT: /kaggle/working is not permanent by itself.')
print('Click Save Version with outputs after expensive runs.')
if not PROCESS_HOLDOUT:
    print('\nDevelopment run protected the holdout.')
    print('When development thresholds are frozen and saved, set PROCESS_HOLDOUT=True for the final three scenes.')

## Final outputs to inspect after the development run

The most important products are:

- `FINAL_EXPORT/FINAL_CLASSIFIED_SAR_CANDIDATES.csv` — all five classes.
- `FINAL_EXPORT/MAIN_VESSEL_RESULTS.csv` — only AIS-matched and non-AIS likely vessels.
- `FINAL_EXPORT/FINAL_CLASSIFIED_SAR_CANDIDATES.geojson` — GIS-ready object points.
- `FINAL_EXPORT/FIGURES/classification_maps/` — final five-class, vessel-only and geographic maps.
- `FINAL_EXPORT/FIGURES/target_property_plots/` — apparent geometry and SAR-contrast QA.
- `09_validation/AIS_CENTERED_DETECTOR_AUDIT.csv` — checks whether AIS-supported vessels had a raw SAR candidate nearby.
- `12_tables/AIS_VS_SAR_GEOMETRY_QA.csv` — candidate-level AIS length vs apparent SAR geometry.
- `10_hard_negatives/HARD_NEGATIVE_CATALOG.csv` — useful island/coast/static negatives for later retraining.
- `SAR_Ship_Project_V4_1_RESULTS_WITH_FIGURES.zip` — compact research output bundle.
- `SAR_Ship_Project_V4_1_PERSIST_ME.zip` — restart/checkpoint bundle for a later holdout run.


In [ ]:

# ============================================================
# V4.1 - CREATE RESEARCH RESULTS ZIP WITH FIGURES
# ============================================================
#
# This ZIP intentionally excludes the very large SAR scene cache.
# The restart/checkpoint ZIP from STEP 30 remains the persistence product.

from pathlib import Path
import zipfile, os

RESULTS_ZIP = WORK_ROOT.parent / 'SAR_Ship_Project_V4_1_RESULTS_WITH_FIGURES.zip'
if RESULTS_ZIP.exists():
    RESULTS_ZIP.unlink()

include_roots = [
    DIRS['final'],
    DIRS['validation'],
    DIRS['tables'],
    DIRS['hard_negatives'],
]

with zipfile.ZipFile(
    RESULTS_ZIP,
    mode='w',
    compression=zipfile.ZIP_DEFLATED,
    allowZip64=True
) as z:
    for root in include_roots:
        if not root.exists():
            continue
        for p in root.rglob('*'):
            if p.is_file():
                arcname = Path(root.name) / p.relative_to(root)
                z.write(p, arcname)

size_mb = RESULTS_ZIP.stat().st_size / (1024**2)

print('='*76)
print('V4.1 RESULTS ZIP CREATED')
print('='*76)
print('Saved as :',RESULTS_ZIP)
print(f'ZIP size : {size_mb:.1f} MB')
print()
print('Contains:')
print(' - final classified CSV/Parquet')
print(' - classification maps')
print(' - vessel-only overlays')
print(' - geographic maps')
print(' - target-property QA plots')
print(' - spatial diagnostic plots')
print(' - GeoJSON')
print(' - validation tables')
print(' - hard-negative catalogue')
print()
print('The large SAR scene cache is NOT duplicated here.')
print('Use SAR_Ship_Project_V4_1_PERSIST_ME.zip for restart/checkpoint persistence.')
